In [ ]:
#| default_exp game/mechanics/space

In [ ]:
import sys
import math
from fastcore.basics import patch
from fasthtml.common import *
from fasthtml.jupyter import *
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from collections import deque , Counter

from monsterui.all import TableT, Card, CardHeader, CardTitle

In [ ]:
from HexMagic.game.data import Settlement, Kingdom, Piece,  GameBoard

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum
from typing import Literal
import heapq, math

import pyomo.environ as pyo
from fasthtml.common import *
from HexMagic.primitives import HexGrid, HexPosition
from HexMagic.game.data import Piece, PieceType, GameBoard

In [ ]:
#| export
import httpx
import random
import pandas as pd

from dataclasses import dataclass
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import math

from dataclasses import dataclass, field
from functools import cached_property

import heapq
import io
from monsterui.all import *
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import FlowOverlay, CreamOverlay, RiverOverlay, ClimateOverlay, OverlayContext
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay

In [ ]:
from HexMagic.game.mechanics.simulator import TurnReport, SimEventType, FoodSimulator, squad_chart, SquadFoodTrailOverlay, apply_food_profile
from HexMagic.game.mechanics.simulator import  knownWorld, smart_placement,optimal_camp_layout, PieceShadow, GameParts,FOOD_PROFILE_V2, _solver_ok
from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay
from HexMagic.game.piece import  _facing_toward, GroupedPieceList, PieceList
from HexMagic.game.flag import GameContext, DiagramGlyphs
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad

## Demo

In [ ]:
myStuff = GameParts()
def rebootWorld():
    myStuff.loadGeology(knownWorld())
myStuff.grid.adjustRadius(40)
rebootWorld()

In [ ]:
showDemo = True
myStuff.grid.adjustRadius(40)

In [ ]:
showDemo = False

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

In [ ]:
!cat ../../../HexMagic/overlay.py

In [ ]:
!cat ../../../HexMagic/game/flag.py

In [ ]:
def _find_hungry(alive, grid, piece, safety_turns=2):
    """Find the most desperate piece, sorted by urgency."""
    hungry = []
    for p in alive:
        if p is piece: continue
        if p.health <= 0: continue
        turns_left = p.food / max(p.diet, 0.1)
        if turns_left < safety_turns * 3:  # worth helping
            hungry.append((p, turns_left))
    hungry.sort(key=lambda h: h[1])  # most desperate first
    return hungry


def _adjacent(grid, loc_a, loc_b):
    """True if same hex or neighbors."""
    if loc_a == loc_b: return True
    return loc_b in grid.neighborsOf(loc_a)


def _donor_surplus(piece, safety_turns=2):
    """How much food this piece can spare."""
    keep = piece.diet * safety_turns
    return max(0, piece.food - keep)

In [ ]:
def food_redistribute_adjacent(report: TurnReport):
    """Tier 1: Direct adjacent transfers. No instructions touched.
    
    Safe to call every turn, even during march.
    Moves food from anyone with surplus to the most desperate neighbor.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    
    # Most desperate first
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        # Donors: adjacent pieces with surplus above safety margin
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break


def food_redistribute_rescue(report: TurnReport):
    """Tiers 2-3: Rotate/move toward hungry. Edits instructions.
    
    Only call during CAMP or emergency. Will disrupt march orders.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # Anyone still desperate after tier-1?
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # tier 1 already handled
        
        # ── Tier 2: Rotate toward hungry ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            rot = Instruction.ROT_R.value if diff <= 3 else Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue
            
            rules.insert(cursor, rot)
            continue
        
        # ── Tier 3: Move toward hungry (only if truly dying) ──
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue
                rules.insert(cursor, Instruction.FORWARD.value)

## 5. Food-Aware Routing

Dijkstra where step cost penalizes barren neighborhoods:

```
step_cost = 1.0 + barren_penalty × (1 - sustainability)
sustainability = min(1.0, best_pair_export / queen_diet)
```

"Best pair export" = top two pawn food sources within `cone_range` of the hex. Routes the queen through corridors where 2 pawns can feed her, avoids barren stretches even if they're shorter.

In [ ]:
def simple_hex_path(start, end, grid, elevations):
    """BFS shortest path on land hexes (ignores food, just walkability)."""
    if start == end: return [start]
    N = len(elevations)
    visited = {start}
    parent = {start: None}
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == end:
            path = []
            while cur is not None:
                path.append(cur); cur = parent[cur]
            return path[::-1]
        for nb in grid.neighborsOf(cur):
            if nb < 0 or nb >= N or nb in visited: continue
            if nb in grid.invalidRegion or elevations[nb] <= 0: continue
            visited.add(nb); parent[nb] = cur; queue.append(nb)
    return None

In [ ]:
def route_to_instructions(route, grid, start_facing):
    """Convert a list of adjacent hex indices into exact FORWARD/ROT instructions."""
    rules = []
    facing = start_facing
    dirs = HexPosition.directions()
    
    for i in range(len(route) - 1):
        # Which of the 6 directions goes from route[i] to route[i+1]?
        target_facing = None
        for d in range(6):
            nb_idx = grid.hexposition_to_index(dirs[d], route[i])
            if nb_idx == route[i + 1]:
                target_facing = d
                break
        
        if target_facing is None:
            print(f"⚠️ {route[i]}→{route[i+1]} not adjacent! Skipping.")
            continue
        
        # Rotate shortest direction
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        
        rules.append(Instruction.FORWARD.value)
    return rules, facing

In [ ]:
def food_aware_route(start_hex, end_hex, grid, elevations, food_tiers,
                     cone_range=4, pawn_harvest=1.3, pawn_diet=0.7,
                     queen_diet=7.0, barren_penalty=10.0, countries=None):
    """Dijkstra pathfinding that prefers hexes near good food sources.
    
    Cost model for each step:
      base_cost = 1.0  (one hex of movement)
      
      For the neighbor hex, look at the best pawn-pair export 
      available within cone_range. The richer the neighborhood,
      the cheaper the step.
      
      cost = base_cost + barren_penalty × (1 - pair_sustainability)
      
      where pair_sustainability = min(1, best_pair_export / queen_diet)
    
    This routes the queen through corridors where 2 pawns can feed her,
    and avoids barren stretches even if they're shorter.
    """
    N = len(elevations)
    
    # ── Precompute: for each hex, best pawn-pair export within cone_range ──
    # First, get per-hex export (what a single pawn stationed there produces)
    hex_export = [0.0] * N
    for i in range(N):
        if i in grid.invalidRegion or elevations[i] <= 0: continue
        tier = int(food_tiers[i]) if 0 <= i < len(food_tiers) else 0
        net = pawn_harvest * tier - pawn_diet
        hex_export[i] = max(0, net)
    
    # For each hex, find the best pair of food sources within cone_range
    # (cached so Dijkstra doesn't recompute)
    _pair_cache = {}
    
    def best_pair_export(hex_idx):
        if hex_idx in _pair_cache: return _pair_cache[hex_idx]
        
        # Collect all food sources within cone_range
        hp_center = grid.index_to_hexposition(hex_idx)
        sources = []
        for ring in range(1, int(cone_range) + 1):
            for hp in hp_center.ring(ring):
                nidx = grid.hexposition_to_index(hp, hex_idx)
                if nidx < 0 or nidx >= N: continue
                if nidx in grid.invalidRegion: continue
                if hex_export[nidx] > 0:
                    sources.append(hex_export[nidx])
        
        # Best pair
        sources.sort(reverse=True)
        if len(sources) >= 2:
            result = sources[0] + sources[1]
        elif len(sources) == 1:
            result = sources[0]
        else:
            result = 0.0
        
        _pair_cache[hex_idx] = result
        return result
    
    # ── Dijkstra ──
    INF = float('inf')
    dist = {start_hex: 0.0}
    parent = {start_hex: None}
    pq = [(0.0, start_hex)]
    
    while pq:
        d, cur = heapq.heappop(pq)
        if cur == end_hex: break
        if d > dist.get(cur, INF): continue
        
        for nb in grid.neighborsOf(cur):
            if nb < 0 or nb >= N: continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            
            # Cost: base + penalty for poor food neighborhood
            pair_exp = best_pair_export(nb)
            sustainability = min(1.0, pair_exp / queen_diet)
            step_cost = 1.0 + barren_penalty * (1.0 - sustainability)
            
            new_dist = d + step_cost
            if new_dist < dist.get(nb, INF):
                dist[nb] = new_dist
                parent[nb] = cur
                heapq.heappush(pq, (new_dist, nb))
    
    if end_hex not in parent:
        print(f"No route ⬡{start_hex} → ⬡{end_hex}"); return None
    
    # Reconstruct
    route = []
    cur = end_hex
    while cur is not None:
        route.append(cur)
        cur = parent[cur]
    route.reverse()
    
    # Stats
    total_cost = dist[end_hex]
    min_pair = min(best_pair_export(h) for h in route)
    avg_pair = sum(best_pair_export(h) for h in route) / len(route)
    barren_count = sum(1 for h in route if best_pair_export(h) < queen_diet)
    
    print(f"Food-aware route: {len(route)} hexes, cost={total_cost:.1f}")
    print(f"  Pair export — min={min_pair:.1f}, avg={avg_pair:.1f}, "
          f"barren={barren_count}/{len(route)}")
    
    return route

## 6. Pawn-Covered Route MIP

Given the food-aware route, find minimum pawns to cover it. Gaps are allowed if bishops can bridge them. Uses soft constraints — the `deficit` variable at each route hex represents food the bishop must supply, bounded by total bishop capacity.

In [ ]:
def pawn_covered_route(start_hex, end_hex, grid, elevations, food_tiers,
                       pawn_harvest=1.3, pawn_diet=0.7, queen_diet=7.0,
                       cone_range=4, barren_penalty=10.0, countries=None,
                       n_bishops=1, bishop_capacity=140.0, bishop_diet=2.5,
                       n_queens=1):
    """Find food-aware route + optimal pawn placements with bishop gap support.
    
    Soft coverage: gaps are allowed if available bishops can ferry the queen
    across them. Slack per route hex represents food deficit that bishops
    must supply. Total slack is bounded by bishop budget.
    
    Args:
        n_bishops:        bishops available for gap crossings
        bishop_capacity:  food capacity per bishop
        bishop_diet:      bishop food per turn while escorting
        n_queens:         number of queens (scales food demand)
    
    Returns dict with route, placements, gaps, queen_diet, bishop_fuel_needed
    """
    effective_diet = queen_diet * n_queens
    
    # ── 1. Food-aware route (Dijkstra with barren penalty) ──
    route = food_aware_route(start_hex, end_hex, grid, elevations,
                             food_tiers, barren_penalty)
    if not route:
        print(f"No route found ⬡{start_hex} → ⬡{end_hex}")
        return None
    
    W = len(route)
    route_set = set(route)
    
    # Route stats
    pair_exp = []
    barren_count = sum(1 for h in route 
                       if h >= len(food_tiers) or food_tiers[h] == 0)
    print(f"Food-aware route: {W} hexes")
    
    # ── 2. Hex distance helper (O(1) via cube coords) ──
    def hex_dist(a, b):
        pa = grid.index_to_hexposition(a)
        pb = grid.index_to_hexposition(b)
        return pa.distance(pb)
    
    # ── 3. Candidate pawn positions ──
    # Find all hexes within cone_range of any route hex, with food > 0
    seen_cands = set()
    candidates = []
    
    for rh in route:
        # BFS outward from route hex to find candidate sites
        frontier = {rh}
        visited_bfs = {rh}
        for _ in range(cone_range + 1):
            nxt = set()
            for h in frontier:
                for nb in grid.neighborsOf(h):
                    if nb < 0 or nb >= len(elevations): continue
                    if nb in visited_bfs: continue
                    if elevations[nb] <= 0: continue
                    visited_bfs.add(nb)
                    nxt.add(nb)
            frontier = nxt
        
        for h in visited_bfs:
            if h in seen_cands or h in route_set: continue
            if h in grid.invalidRegion: continue
            
            tier = int(food_tiers[h]) if h < len(food_tiers) else 0
            if tier <= 0: continue
            
            export = tier * pawn_harvest - pawn_diet
            if export <= 0: continue
            
            # Coverage: route indices this candidate can feed
            coverage = set()
            for w, rw in enumerate(route):
                if hex_dist(h, rw) <= cone_range:
                    coverage.add(w)
            if not coverage: continue
            
            candidates.append({
                'hex': h, 'tier': tier, 'export': export, 'coverage': coverage,
            })
            seen_cands.add(h)
    
    if not candidates:
        print("No pawn candidates found!")
        return None
    
    S = range(len(candidates))
    W_range = range(W)
    
    # Pre-compute (candidate, route_hex) → export
    export_at = {}
    for s in S:
        for w in candidates[s]['coverage']:
            export_at[s, w] = candidates[s]['export']
    
    # Identify fully uncoverable hexes (no candidate reaches them at all)
    uncoverable = set()
    for w in W_range:
        if not any((s, w) in export_at for s in S):
            uncoverable.add(w)
    
    if uncoverable:
        print(f"  {len(uncoverable)} route hexes unreachable by any pawn "
              f"(indices: {sorted(uncoverable)[:10]}{'...' if len(uncoverable) > 10 else ''})")
    
    # ── 4. MIP with soft constraints ──
    m = pyo.ConcreteModel()
    m.S = pyo.Set(initialize=list(S))
    m.W = pyo.Set(initialize=list(W_range))
    
    # Binary: place this candidate?
    m.placed = pyo.Var(m.S, domain=pyo.Binary)
    
    # Continuous: food deficit at each route hex (bishop must supply)
    m.deficit = pyo.Var(m.W, domain=pyo.NonNegativeReals,
                      bounds=(0, effective_diet))
    
    # Binary: is this route hex a gap? (slack > 0)
    m.is_gap = pyo.Var(m.W, domain=pyo.Binary)
    
    # ── Coverage: placed pawn exports + slack >= queen demand ──
    m.cover = pyo.ConstraintList()
    for w in m.W:
        pawn_supply = sum(export_at[s, w] * m.placed[s]
                          for s in m.S if (s, w) in export_at)
        m.cover.add(pawn_supply + m.deficit[w] >= effective_diet)
    
    # ── Link slack ↔ gap indicator ──
    m.gap_link = pyo.ConstraintList()
    for w in m.W:
        m.gap_link.add(m.deficit[w] <= effective_diet * m.is_gap[w])
    
    # ── Bishop budget: food to supply + bishop self-consumption ≤ capacity ──
    # sum(slack) = total food bishops must give the queen
    # bishop_diet × num_gap_hexes = food bishops eat while in gaps
    bishop_budget = n_bishops * bishop_capacity
    m.bishop_budget = pyo.Constraint(
        expr=(sum(m.deficit[w] for w in m.W)
              + bishop_diet * sum(m.is_gap[w] for w in m.W)
              <= bishop_budget))
    
    # ── Objective: minimize pawns, then minimize slack ──
    # Pawns are expensive (weight=1), slack is cheap (weight=0.01)
    # → solver prefers to use bishops for gaps rather than adding marginal pawns
    slack_weight = 0.01
    m.obj = pyo.Objective(
        expr=(sum(m.placed[s] for s in m.S)
              + slack_weight * sum(m.deficit[w] for w in m.W)),
        sense=pyo.minimize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        print(f"MIP infeasible — {len(uncoverable)} uncoverable, "
              f"bishop budget={bishop_budget:.0f}")
        return None
    
    # ── 5. Extract results ──
    placements = []
    for s in m.S:
        if pyo.value(m.placed[s]) > 0.5:
            placements.append(candidates[s])
    placements.sort(key=lambda p: min(p['coverage']))
    
    gaps = sorted(w for w in W_range if pyo.value(m.deficit[w]) > 0.01)
    gap_hexes = [route[w] for w in gaps]
    bishop_fuel = sum(pyo.value(m.deficit[w]) for w in W_range)
    bishop_turns = sum(1 for w in W_range if pyo.value(m.is_gap[w]) > 0.5)
    
    # ── Per-hex export summary ──
    min_exp = float('inf')
    total_exp = 0
    pawn_covered = 0
    for w in W_range:
        exp = sum(p['export'] for p in placements if w in p['coverage'])
        total_exp += exp
        if exp >= effective_diet: pawn_covered += 1
        min_exp = min(min_exp, exp)
    avg_exp = total_exp / W if W > 0 else 0
    
    print(f"  Pair export — min={min_exp:.1f}, avg={avg_exp:.1f}, "
          f"barren={W - pawn_covered}/{W}")
    print(f"PCR: route {W} hexes, {len(placements)} pawns, "
          f"{pawn_covered}/{W} pawn-covered"
          + (f", {len(gaps)} gap hexes → bishop" if gaps else " ✅"))
    
    for p in placements:
        cmin, cmax = min(p['coverage']), max(p['coverage'])
        print(f"  ⬡{p['hex']} T{p['tier']} export={p['export']:.1f}/t  "
              f"path[{cmin}..{cmax}]")
    
    if gaps:
        print(f"  🐪 Bishop gap: route{gaps} (⬡{gap_hexes})")
        print(f"     fuel={bishop_fuel:.1f} + bishop_eat={bishop_diet * bishop_turns:.1f} "
              f"= {bishop_fuel + bishop_diet * bishop_turns:.1f}"
              f"/{bishop_budget:.0f} "
              f"({'✅' if bishop_fuel + bishop_diet * bishop_turns <= bishop_budget else '❌'})")
    
    return {
        'route': route,
        'placements': placements,
        'gaps': gaps,
        'gap_hexes': gap_hexes,
        'queen_diet': effective_diet,
        'bishop_fuel_needed': bishop_fuel,
        'bishop_turns': bishop_turns,
    }

## Station Segmentation MIP (`find_stations_v3`)

Given a PCR result, optimally assign pawns to cover the route. Gaps go to bishops. Unlike v2 (greedy pair enumeration), this directly optimizes which candidates to activate and naturally handles 1/2/3-pawn coverage per hex. Returns a list of segments — stations (pawn-covered) and caravans (bishop gaps) — in route order.

In [ ]:
@dataclass
class Segment:
    """A contiguous stretch of a corridor — either pawn-covered (station) or bishop gap (caravan)."""
    type: Literal['station', 'caravan']
    route_start: int        # first route index in this segment
    route_end: int          # last route index (inclusive)

    # ── Rendering ──
    style: StyleCSS = None
    line_width: float = 2.0
    dash: str = None        # e.g. "8,4" for caravans

    # ── Station fields (pawn-covered stretch) ──
    hexes: list[int] = field(default_factory=list)      # pawn hex positions
    tiers: list[int] = field(default_factory=list)
    exports: list[float] = field(default_factory=list)
    total_export: float = 0.0
    n_pawns: int = 0

    # ── Caravan fields (bishop gap) ──
    gap_hexes: list[int] = field(default_factory=list)
    gap_len: int = 0
    load_w: int = 0         # route index where bishop loads food
    dock_w: int = 0         # route index where bishop docks
    load_hex: int = 0
    dock_hex: int = 0
    est_turns: int = 0
    fuel_needed: float = 0.0
    burn_rate: float = 0.0

    @property
    def span(self): return self.route_end - self.route_start + 1

    @property
    def label(self):
        "Short label for display."
        return f"{self.type}[{self.route_start}..{self.route_end}]"

    def summary(self):
        if self.type == 'station':
            exp_str = '+'.join(f'{e:.1f}' for e in self.exports)
            return (f"🏕  Station: ⬡{self.hexes} T{self.tiers} "
                    f"({self.n_pawns} pawns) export={exp_str}={self.total_export:.1f} "
                    f"route[{self.route_start}..{self.route_end}]")
        else:
            return (f"🐪 Caravan: route[{self.route_start}..{self.route_end}] "
                    f"({self.gap_len}hex ~{self.est_turns}t) "
                    f"fuel={self.fuel_needed:.1f}")

In [ ]:
@patch
def __ft__(self: Segment):
    is_station = self.type == 'station'
    icon = '🏕' if is_station else '🐪'
    label = 'Station' if is_station else 'Caravan'
    span_txt = f"route[{self.route_start}..{self.route_end}] ({self.span} hex)"

    if is_station:
        rows = []
        for h, t, e in zip(self.hexes, self.tiers, self.exports):
            rows.append(Tr(Td(f'⬡{h}'), Td(f'T{t}'), Td(f'+{e:.1f}/t')))
        body = Div(
            P(Small(span_txt), cls='uk-text-muted'),
            Table(
                Thead(Tr(Th('Hex'), Th('Tier'), Th('Export'))),
                Tbody(*rows),
                cls=(TableT.sm, TableT.divider),
            ) if rows else None,
            P(Strong(f'Total: {self.total_export:.1f}/t'),
              Small(f' ({self.n_pawns} pawns)'), cls='uk-text-right'),
        )
        color = 'success'
    else:
        body = Div(
            P(Small(span_txt), cls='uk-text-muted'),
            Dl(
                Dt('Gap'),      Dd(f'{self.gap_len} hexes  ⬡{self.gap_hexes[:4]}{"…" if len(self.gap_hexes) > 4 else ""}'),
                Dt('Load→Dock'),Dd(f'⬡{self.load_hex} → ⬡{self.dock_hex}'),
                Dt('Fuel'),     Dd(f'{self.fuel_needed:.1f} ({self.burn_rate:.1f}/t × ~{self.est_turns}t)'),
                cls='uk-description-list',
            ),
        )
        color = 'warning'

    return Card(
        CardHeader(CardTitle(f'{icon} {label}', cls=f'uk-text-{color}')),
        body,
        cls='uk-card-small',
    )


In [ ]:
def find_stations_v3(pcr_result, n_pawns=4, n_bishops=1,
                     queen_diet=None, profile=None):
    """MIP: optimally assign pawns to cover the route, gaps go to bishops.

    Returns list of Segment objects in route order.
    """
    p = profile or FOOD_PROFILE_V2
    queen_diet = queen_diet or pcr_result.get('queen_diet', p['queen_diet'])
    b_cap = p.get('bishop_capacity', 140)
    b_diet = p.get('bishop_diet', 3.5)
    burn_rate = queen_diet + b_diet

    placements = pcr_result['placements']
    route = pcr_result['route']
    W = len(route)
    N = len(placements)

    if N == 0:
        print("No PCR candidates!"); return []

    # Pre-compute: for each (candidate, route_hex), the export
    export_at = {}
    for c in range(N):
        for w in placements[c]['coverage']:
            export_at[c, w] = placements[c]['export']

    # ── MIP ──
    m = pyo.ConcreteModel()
    m.C = pyo.RangeSet(0, N - 1)
    m.W = pyo.RangeSet(0, W - 1)

    m.place = pyo.Var(m.C, domain=pyo.Binary)
    m.gap = pyo.Var(m.W, domain=pyo.Binary)

    m.budget = pyo.Constraint(
        expr=sum(m.place[c] for c in m.C) <= n_pawns)

    big_M = queen_diet * 2
    m.cover = pyo.ConstraintList()
    for w in m.W:
        supply = sum(export_at[c, w] * m.place[c]
                     for c in m.C if (c, w) in export_at)
        m.cover.add(supply + big_M * m.gap[w] >= queen_diet)

    max_gap_hexes = int(b_cap * n_bishops / burn_rate) if n_bishops > 0 else 0
    m.bishop_budget = pyo.Constraint(
        expr=sum(m.gap[w] for w in m.W) <= max_gap_hexes)

    m.obj = pyo.Objective(
        expr=sum(m.gap[w] for w in m.W) * 100
             + sum(m.place[c] for c in m.C),
        sense=pyo.minimize)

    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, load_solutions=False)

    if not _solver_ok(result):
        print(f"Stations v3 MIP infeasible: {N} candidates, "
              f"{n_pawns} pawns, {n_bishops} bishops")
        return []

    m.solutions.load_from(result)

    # ── Extract placed pawns ──
    placed = [placements[c] for c in m.C if pyo.value(m.place[c]) > 0.5]

    # ── Classify each route hex ──
    hex_type = ['gap' if pyo.value(m.gap[w]) > 0.5 else 'covered'
                for w in range(W)]

    # ── Group into contiguous Segment objects ──
    segments = []
    i = 0
    while i < W:
        kind = hex_type[i]
        j = i
        while j < W and hex_type[j] == kind:
            j += 1

        if kind == 'covered':
            stretch = set(range(i, j))
            serving = [p for p in placed if p['coverage'] & stretch]
            segments.append(Segment(
                type='station',
                route_start=i,
                route_end=j - 1,
                hexes=[p['hex'] for p in serving],
                tiers=[p['tier'] for p in serving],
                exports=[p['export'] for p in serving],
                total_export=sum(p['export'] for p in serving),
                n_pawns=len(serving),
            ))
        else:
            gap_len = j - i
            est_turns = max(1, (gap_len + 1) // 2)
            load_w = i - 1 if i > 0 else 0
            dock_w = j if j < W else W - 1
            segments.append(Segment(
                type='caravan',
                route_start=i,
                route_end=j - 1,
                gap_hexes=route[i:j],
                gap_len=gap_len,
                load_w=load_w,
                dock_w=dock_w,
                load_hex=route[load_w],
                dock_hex=route[dock_w],
                est_turns=est_turns,
                fuel_needed=round(est_turns * burn_rate, 1),
                burn_rate=burn_rate,
            ))
        i = j

    # ── Report ──
    n_sta = sum(1 for s in segments if s.type == 'station')
    n_car = sum(1 for s in segments if s.type == 'caravan')
    gap_total = sum(s.gap_len for s in segments if s.type == 'caravan')
    pawns_used = len(placed)

    print(f"Stations v3: {pawns_used}/{n_pawns} pawns → "
          f"{n_sta} stations + {n_car} caravans ({W} hex route)")

    for seg in segments:
        print(f"  {seg.summary()}")

    if n_car > 0:
        total_fuel = sum(s.fuel_needed for s in segments if s.type == 'caravan')
        budget = n_bishops * b_cap
        print(f"\n  Gap total: {gap_total} hexes, "
              f"fuel={total_fuel:.1f}/{budget:.0f} "
              f"{'✅' if total_fuel <= budget else '❌ over budget'}")
    else:
        print(f"  Full pawn coverage ✅")

    return segments

## 8. Lighthouse Placement MIP

Facility-location problem for bishops. Candidates are hexes within sight of gap hexes. Each bishop parks at one candidate and covers gap hexes in its sight cone. Fuel constraint limits service turns. Objective: minimize bishops used, tiebreak by elevation (higher = better visibility) and off-route preference.

In [ ]:
def optimal_lighthouse_placement(gap_indices, route, grid, elevations,
                                 n_bishops=1, bishop_sight=4,
                                 queen_diet=7.0, bishop_diet=3.5,
                                 bishop_capacity=140.0,
                                 max_ring=2, profile=None):
    """MIP: place bishops as lighthouses to cover route gap hexes.
    
    Facility location problem:
      - Candidates: hexes within max_ring of any gap hex, on land
      - Each bishop parks at a candidate and covers gap hexes in sight range
      - Fuel constraint: bishop burns (queen_diet + bishop_diet) per turn
        the queen is in its cone, bounded by bishop_capacity
      - Objective: minimize bishops, tiebreak by elevation (higher = better)
    
    Args:
        gap_indices:  list of route indices that need bishop coverage
        route:        full route hex list
        grid:         HexGrid
        elevations:   elevation array
        n_bishops:    max bishops available
        bishop_sight: base sight range
        queen_diet, bishop_diet, bishop_capacity: food profile values
        max_ring:     how far from gap hexes to search for lighthouse sites
        profile:      optional FOOD_PROFILE_V2 override
    
    Returns dict with placements, coverage, fuel info, or None if infeasible.
    """
    if profile:
        queen_diet = profile.get('queen_diet', queen_diet)
        bishop_diet = profile.get('bishop_diet', bishop_diet)
        bishop_capacity = profile.get('bishop_capacity', bishop_capacity)
        bishop_sight = profile.get('pawn_sight', bishop_sight)  # or a bishop_sight key
    
    burn_per_turn = queen_diet + bishop_diet
    gap_set = set(gap_indices)
    gap_hexes = [route[w] for w in gap_indices]
    route_set = set(route)
    G = len(gap_indices)
    
    if G == 0:
        return {'placements': [], 'fuel': {}, 'total_fuel': 0, 'feasible': True}
    
    # ── Candidate lighthouse hexes ──
    seen = set()
    candidates = []
    
    for gh in gap_hexes:
        hp_g = grid.index_to_hexposition(gh)
        for ring in range(0, bishop_sight + max_ring + 1):
            ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
            for hp in ring_hps:
                idx = grid.hexposition_to_index(hp, gh)
                if idx < 0 or idx in seen: continue
                if idx in grid.invalidRegion: continue
                if idx >= len(elevations) or elevations[idx] <= 0: continue
                seen.add(idx)
                
                # Which gap hexes can this candidate see?
                hp_c = grid.index_to_hexposition(idx)
                can_see = []
                for wi, w in enumerate(gap_indices):
                    hp_r = grid.index_to_hexposition(route[w])
                    if hp_c.distance(hp_r) <= bishop_sight:
                        can_see.append(wi)  # index into gap_indices
                
                if not can_see: continue
                
                candidates.append({
                    'hex': idx,
                    'can_see': can_see,  # indices into gap_indices
                    'elevation': elevations[idx],
                    'on_route': idx in route_set,
                })
    
    if not candidates:
        print("No lighthouse candidates found!")
        return None
    
    C = len(candidates)
    B = n_bishops
    
    # ── Pyomo MIP ──
    m = pyo.ConcreteModel()
    m.C = pyo.RangeSet(0, C - 1)
    m.G = pyo.RangeSet(0, G - 1)
    m.B = pyo.RangeSet(0, B - 1)
    
    # place[b, c]: bishop b placed at candidate c?
    m.place = pyo.Var(m.B, m.C, domain=pyo.Binary)
    
    # serve[b, g]: bishop b feeds queen at gap index g?
    m.serve = pyo.Var(m.B, m.G, domain=pyo.Binary)
    
    # ── Each bishop placed at most one candidate ──
    m.one_site = pyo.Constraint(m.B, rule=lambda m, b:
        sum(m.place[b, c] for c in m.C) <= 1)
    
    # ── Every gap hex must be served by at least one bishop ──
    m.covered = pyo.Constraint(m.G, rule=lambda m, g:
        sum(m.serve[b, g] for b in m.B) >= 1)
    
    # ── Can only serve from a placed bishop within sight ──
    # Pre-compute: which (candidate, gap_index) pairs are valid
    valid_cg = set()
    for c_idx, cand in enumerate(candidates):
        for g_local in cand['can_see']:
            valid_cg.add((c_idx, g_local))
    
    m.sight = pyo.ConstraintList()
    for b in m.B:
        for g in m.G:
            # serve[b,g] ≤ Σ place[b,c] for candidates c that can see gap g
            covers = [c for c in m.C if (c, g) in valid_cg]
            if not covers:
                m.sight.add(m.serve[b, g] == 0)
            else:
                m.sight.add(m.serve[b, g] <= sum(m.place[b, c] for c in covers))
    
    # ── Fuel constraint: each bishop's total service ≤ capacity ──
    # Each gap hex served costs burn_per_turn (1 turn per hex at queen speed)
    m.fuel = pyo.Constraint(m.B, rule=lambda m, b:
        sum(m.serve[b, g] for g in m.G) * burn_per_turn <= bishop_capacity)
    
    # ── Objective: minimize bishops used, tiebreak by elevation ──
    # "bishop used" = placed anywhere
    m.used = pyo.Var(m.B, domain=pyo.Binary)
    m.link_used = pyo.Constraint(m.B, rule=lambda m, b:
        m.used[b] >= sum(m.place[b, c] for c in m.C))
    
    # Elevation bonus (normalized small so it's just a tiebreak)
    max_elev = max(c['elevation'] for c in candidates) or 1
    elev_bonus = sum(
        m.place[b, c] * (candidates[c]['elevation'] / max_elev) * 0.01
        for b in m.B for c in m.C
    )
    # Prefer off-route (small penalty for on-route)
    route_penalty = sum(
        m.place[b, c] * (0.001 if candidates[c]['on_route'] else 0)
        for b in m.B for c in m.C
    )
    
    m.obj = pyo.Objective(
        expr=sum(m.used[b] for b in m.B) - elev_bonus + route_penalty,
        sense=pyo.minimize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    
    if not _solver_ok(result):
        print(f"Lighthouse MIP infeasible: {G} gap hexes, {C} candidates, {B} bishops")
        return None
    
    # ── Extract results ──
    placements = []
    fuel_usage = {}
    
    for b in m.B:
        for c in m.C:
            if pyo.value(m.place[b, c]) > 0.5:
                served = [g for g in m.G if pyo.value(m.serve[b, g]) > 0.5]
                served_route_w = [gap_indices[g] for g in served]
                fuel = len(served) * burn_per_turn
                
                placements.append({
                    'bishop': b,
                    'hex': candidates[c]['hex'],
                    'elevation': candidates[c]['elevation'],
                    'on_route': candidates[c]['on_route'],
                    'serves_gap': served,
                    'serves_route_w': served_route_w,
                    'fuel_used': round(fuel, 1),
                    'fuel_remaining': round(bishop_capacity - fuel, 1),
                })
                fuel_usage[b] = fuel
    
    total_fuel = sum(fuel_usage.values())
    bishops_used = len(placements)
    
    # ── Report ──
    print(f"Lighthouse MIP: {bishops_used}/{B} bishops cover {G} gap hexes")
    for p in placements:
        w_range = f"route[{min(p['serves_route_w'])}..{max(p['serves_route_w'])}]"
        print(f"  🔦 Bishop {p['bishop']}: ⬡{p['hex']} "
              f"(elev={p['elevation']:.0f}"
              f"{', ON route' if p['on_route'] else ''}) "
              f"serves {len(p['serves_gap'])} hexes {w_range} "
              f"fuel={p['fuel_used']}/{bishop_capacity:.0f} "
              f"(margin={p['fuel_remaining']})")
    
    return {
        'placements': placements,
        'fuel': fuel_usage,
        'total_fuel': round(total_fuel, 1),
        'bishops_used': bishops_used,
        'burn_per_turn': burn_per_turn,
        'feasible': True,
    }

## 9. Corridor Types & Data Structures

These dataclasses are the "nouns" of the system — they carry the planning state between pipeline stages.

In [ ]:
class CorridorType(Enum):
    PERMANENT = 'PERM'   # established route, kept between turns
    TEMPORARY = 'TEMP'


@dataclass
class Target:
    "Where a queen wants to go and why."
    hex:      int
    purpose:  str          # 'attack' | 'settle' | 'harvest' | 'explore'
    priority: float        = 1.0
    deadline: int | None   = None
    payoff:   float        = 0.0
    style:    StyleCSS     = None
    glyph:    object       = None

    def __ft__(self):
        dl = f'{self.deadline}t' if self.deadline else '—'
        return Tr(
            Td(f'⬡{self.hex}'), Td(self.purpose),
            Td(f'{self.priority:.1f}'), Td(dl), Td(f'{self.payoff:.1f}'),
        )


@dataclass
class Assignment:
    "Queen → Target → Corridor — output of plan()."
    queen:           Piece
    target:          Target
    corridor:        'Corridor'
    turns_to_arrive: int = 0

    def __ft__(self):
        return Tr(
            Td(self.queen.name),
            Td(f'⬡{self.target.hex}'),
            Td(self.target.purpose),
            Td(f'{self.turns_to_arrive}t'),
            Td(f'⬡{self.corridor.start}→⬡{self.corridor.end}'),
            Td('✅' if self.corridor.feasible else '❌'),
        )



    

In [ ]:
@dataclass
class Corridor:
    hexes:           list[int]        = field(default_factory=list)
    kind:            CorridorType     = CorridorType.PERMANENT
    segments:        list[Segment]    = field(default_factory=list)
    n_stations:      int              = 0
    n_gaps:          int              = 0
    fuel_needed:     float            = 0.0
    bishop_min_fuel: float            = 0.0
    duration:        int              = 0
    congestion:      int              = 0
    valid:           bool             = True
    style:           StyleCSS         = None
    target:          Target           = None
    glyph:           DiagramGlyphs    = None

    @property
    def feasible(self):
        if not self.valid: return False
        if self.n_gaps == 0: return True
        return self.bishop_min_fuel >= self.fuel_needed

    @property
    def start(self):  return self.hexes[0]  if self.hexes else None
    @property
    def end(self):    return self.hexes[-1] if self.hexes else None
    @property
    def length(self): return len(self.hexes)

   
    @property
    def label(self):
        "Short label for display."
        return f"{self.kind.value}:⬡{self.start}→⬡{self.end}"

   
    def __len__(self):
        return len(self.hexes)

    def __getitem__(self, w):
        """corridor[w] → Segment covering route index w, or KeyError."""
        for seg in self.segments:
            if seg.route_start <= w <= seg.route_end:
                return seg
        raise KeyError(w)

    def __contains__(self, hex_idx):
        """⬡idx in corridor → True if hex is on the route."""
        return hex_idx in self.hexes

    def __iter__(self):
        """Iterate segments in route order."""
        return iter(self.segments)

    def index(self, hex_idx):
        """Map hex index → route position, or ValueError."""
        try:
            return self.hexes.index(hex_idx)
        except ValueError:
            raise ValueError(f"⬡{hex_idx} not on this corridor")
    





the route index and segement_at feels like we should be using some sort of __listthing__ to access the segements as if this were a list

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great instinct! `segment_at(w)` is just `corridor[w]`, and `route_index` is `__contains__` / `index`. You can make `Corridor` act like a sequence of segments keyed by route index with a few dunders:

```python
def __len__(self):
    return len(self.hexes)

def __getitem__(self, w):
    """corridor[w] → Segment covering route index w, or KeyError."""
    for seg in self.segments:
        if seg.route_start <= w <= seg.route_end:
            return seg
    raise KeyError(w)

def __contains__(self, hex_idx):
    """⬡idx in corridor → True if hex is on the route."""
    return hex_idx in self.hexes

def __iter__(self):
    """Iterate segments in route order."""
    return iter(self.segments)

def index(self, hex_idx):
    """Map hex index → route position, or ValueError."""
    try:
        return self.hexes.index(hex_idx)
    except ValueError:
        raise ValueError(f"⬡{hex_idx} not on this corridor")
```

Now you get natural usage like:

```python
seg = corridor[w]          # instead of corridor.segment_at(w)
w   = corridor.index(h)    # instead of corridor.route_index(h)
if h in corridor: ...      # membership test
for seg in corridor: ...   # iterate segments
```

Also — quick bug: your current `route_index`, `segment_at`, and `label` are all missing `self` as the first parameter. The `@property` on `label` won't work either since it's not receiving `self`. The dunder approach above fixes all of that naturally.

In [ ]:
@dataclass
class CrossingPlan:
    corridor:         Corridor
    roster:           list[dict]       = field(default_factory=list)
    queen_start_turn: int              = 0
    schedule:         pd.DataFrame     = None
    feasible:         bool             = False
    queen_arrival:    int              = 0
    food_margin:      float            = 0.0
    bishop_fuel_used: float            = 0.0

    def pieces_needed(self):
        from collections import Counter
        return Counter(r['piece_type'] for r in self.roster if r['role'] != 'queen')

In [ ]:
@patch
def __ft__(self: Corridor):
    status = '✅' if self.feasible else '❌'
    kind_badge = 'primary' if self.kind == CorridorType.PERMANENT else 'muted'

    header = Div(
        Span(f'⬡{self.start} → ⬡{self.end}', cls='uk-text-bold'),
        Span(f' ({self.length} hex)', cls='uk-text-muted'),
        Span(f' {self.kind.value}', cls=f'uk-label uk-label-{kind_badge} uk-margin-small-left'),
        Span(f' {status}', cls='uk-margin-small-left'),
    )

    stats = Div(
        Small(f'{self.n_stations} stations · {self.n_gaps} gaps'),
        Small(f' · fuel {self.fuel_needed:.0f}/{self.bishop_min_fuel:.0f}') if self.n_gaps else None,
        cls='uk-text-muted',
    )

    seg_cards = Div(*[s.__ft__() for s in self.segments],
                    cls='uk-grid uk-child-width-1-2@m uk-grid-small') if self.segments else P(
        Em('No segments'), cls='uk-text-muted')

    return Card(
        header, stats, seg_cards,
        cls='uk-card-small',
    )


In [ ]:
@patch
def __ft__(self: CrossingPlan):
    needed = self.pieces_needed()
    summary = ', '.join(f"{v}× {k.name}" for k, v in needed.items())
    status = '✅' if self.feasible else '❌'

    crossing_info = Table(Tbody(
        Tr(Td(B('Roster')),      Td(summary or 'queen only')),
        Tr(Td(B('Queen go')),    Td(f'turn {self.queen_start_turn}')),
        Tr(Td(B('Arrival')),     Td(f'turn {self.queen_arrival}')),
        Tr(Td(B('Food min')),    Td(f'{self.food_margin:.1f}')),
        Tr(Td(B('Bishop fuel')), Td(f'{self.bishop_fuel_used:.0f}')),
    ), cls=TableT.sm)

    timetable = schedule_timetable(self.schedule, title='Crossing Schedule')

    return Div(
        Card(crossing_info, header=CardTitle(f'Crossing Plan {status}')),
        self.corridor.__ft__(),
        timetable,
    )


should corridors be something we do in combination with pawn covers/ stations? Corriods might be too plain on its own.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, I think you're right — a bare polyline doesn't tell you much. A **CorridorOverlay** that shows the route *plus* station/gap coloring in one go would be much more informative. Something like:

- **Covered stretches** — solid colored path segments (or tinted hex fills)
- **Gap stretches** — dashed or differently-colored segments
- **Station pawns** — small markers at pawn hexes, maybe with a subtle coverage cone

That way one `OverlaySpec` takes a corridor (or `CrossingPlan`) and renders the whole story. Lighthouses could still be a separate overlay since they're bishop-specific and layered on top.

Does that split feel right — one combined **CorridorOverlay** (route + stations + gaps) and a separate **LighthouseOverlay**?

I think that is right.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's start with the **CorridorOverlay** then. First decision: what data does the renderer need?

Looking at your `find_stations_v3` output, each segment already has `type` ('station' or 'caravan'), the route index range (`min_w`/`max_w` or `gap_start`/`gap_end`), and pawn hex positions. So the overlay just needs:

- The **route** hex list (to get center coords for the path)
- The **segments** list (to color station vs caravan stretches)

That means the factory signature would be something like:

```python
def CorridorOverlay(corridor, route, **kw) -> OverlaySpec:
```

where `corridor` is the segments list and `route` is the hex index list. The renderer would loop through segments, draw solid path chunks for stations and dashed ones for caravans, then drop small markers at pawn hexes.

Does passing the data in directly (like `RiverOverlay` does with its params) feel right, or would you rather pull it from `ctx.extras` (like `board` or `basins`)?

So parmater are meant to be settings - ocean colors, river width, ctx is meant to have data

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Got it — that's consistent with the rest of your overlays. So `CorridorOverlay` factory params would be things like `station_color`, `gap_dash`, `pawn_marker_size`, and the actual corridor + route data comes from `ctx.extras`.

That means the corridor data needs to get into the context. Two natural options:

1. **Attach it to `board`** — e.g. `board.corridors` — since corridors are game state
2. **Pass it as a new extra** — `TerrainDisplay(..., corridors=my_corridors)` and wire it through like `basins`

Which feels more natural given how your `GameBoard` / `GameParts` is structured?

one aside on corridors is we might want to have who controls it by country flag.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — that's a great idea. You already have `country.flag.kingStyle()` generating per-kingdom styles in `countries_overlay`. So a corridor could carry a reference to its owning `Kingdom` (or just its `CountryFlag`), and the renderer would use that to color the route segments.

That suggests corridors should live on the `Kingdom` or `GameBoard` level rather than as a standalone extra. Something like `kingdom.corridors` — each corridor knows its owner, and the overlay iterates kingdoms to draw their corridors in flag colors.

Does a corridor belong to a single kingdom, or could it be shared (e.g. a trade route between two)?

In [ ]:
!cat ../../../HexMagic/weather.py

Do we want to have some sort of field nparray kind of thing for these corridors, stations idea?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Interesting question. Your `terrain.fields` pattern (per-hex numpy arrays) works brilliantly for continuous data — temperature, precipitation, elevation. But corridors and stations are **sparse and structured** — a corridor is an ordered sequence of hexes with typed segments and metadata. That doesn't map naturally to a flat array.

I'd say the corridors themselves should be **structured objects** (the `Corridor` / segments you already have), living on the kingdom or board. But you could derive **per-hex arrays** for rendering efficiency, like:

- `corridor_density[hex]` — how many corridors cross this hex (congestion heatmap)
- `food_coverage[hex]` — total pawn export available at this hex

Similar to how `DrainageBasins` is a structured object but you query it per-hex when rendering.

So: objects for the data, optional derived arrays for overlays that need per-hex lookups. Does that split make sense for your use case, or are you thinking of a scenario where a flat array would be the primary storage?

ok. lets used structured, I guess where I am at is that we might want to be able to pass in styles for different corridors and stations. The overlay doesn't need to know how these styles are applied. I guess one option is we could add CSSStle to these objects which the overlay could then use.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a really clean approach — it matches how your hex system already works. Each `Hex` carries a `style`, and the renderer just uses it. Same idea here: the corridor/segment carries its `StyleCSS`, and the overlay renders without caring *why* that style was chosen.

The style assignment logic (kingdom flag colors, permanent vs temporary, station vs caravan) lives wherever the corridors are created — maybe a method on `Kingdom` or `GameBoard` that stamps the styles. The overlay is purely: "give me styled objects, I'll draw them."

So you'd add a `style: StyleCSS` field to `Corridor` and to each segment dict (or promote segments to a dataclass with a `style` field). Does that sound right, or would you want style only at the corridor level and derive segment styles from it?

Ok I added style and glyphs to targets and style to corriodrs and segments. These would all be optional

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice. So the fallback chain for rendering would naturally be:

- **Segment** style → if `None`, fall back to **Corridor** style → if `None`, use overlay defaults
- **Target** gets its own style + glyph for endpoint markers (independent of corridor)

That gives you full flexibility — a kingdom can stamp its flag style on the corridor, individual segments can override (e.g. red dashes for caravans), and targets can have distinct markers regardless.

Ready to write the `CorridorOverlay` factory? The renderer would:

1. Loop corridors from `ctx.board` (or wherever they land)
2. For each corridor, walk its segments, draw path chunks with the resolved style
3. Drop pawn markers at station hexes
4. Optionally draw target glyphs at endpoints

What should the overlay pull corridors from — `ctx.board.corridors`, `ctx.board.kingdoms[i].corridors`, or a flat list in `ctx.extras`?

I think a flat list in extras. we are going to want to customize which coridors we are showing

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Perfect — maximum flexibility. You filter the list before passing it in, the overlay doesn't care.

Here's a first cut:

```python
def CorridorOverlay(
    default_color='#2c3e50',
    gap_color='#e74c3c',
    gap_dash='8,4',
    station_color='#27ae60',
    pawn_radius=4,
    path_width=3.0,
    show_pawns=True,
    show_targets=True,
    target_radius=8,
    **kw
) -> OverlaySpec:
    """Render corridors with station/caravan segments and optional pawn markers."""

    def _resolve_style(segment, corridor, builder, default_name, default_fill,
                       dash=None, width=path_width):
        """Segment style → corridor style → generated default."""
        style = segment.style or (corridor.style if hasattr(corridor, 'style') else None)
        if style:
            builder.add_style(style)
            return style
        s = StyleCSS(default_name, fill='none', stroke=default_fill,
                     stroke_width=width, opacity=0.85)
        if dash:
            s.properties['stroke-dasharray'] = dash
        builder.add_style(s)
        return s

    def render(ctx: OverlayContext) -> str:
        corridors = getattr(ctx, 'corridors', None)
        if not corridors:
            return ''

        grid = ctx.grid
        builder = ctx.builder
        parts = []

        for ci, corridor in enumerate(corridors):
            route = corridor.hexes
            if not route:
                continue

            segs = getattr(corridor, 'segments', None) or []

            if not segs:
                # No segments — draw whole route with corridor style
                style = corridor.style
                if not style:
                    style = StyleCSS(f'corridor_{ci}', fill='none',
                                     stroke=default_color, stroke_width=path_width,
                                     opacity=0.85)
                builder.add_style(style)
                points = [grid.hexes[h].center for h in route
                          if 0 <= h < len(grid.hexes)]
                if len(points) >= 2:
                    path = MapPath(points, style)
                    parts.append(path.drawSpline(adds=f'class="{style.name}"'))
                continue

            # Draw each segment
            for si, seg in enumerate(segs):
                w_start = seg.route_start
                w_end = seg.route_end
                seg_hexes = route[w_start:w_end + 1]

                points = [grid.hexes[h].center for h in seg_hexes
                          if 0 <= h < len(grid.hexes)]
                if len(points) < 2:
                    continue

                if seg.type == 'station':
                    style = _resolve_style(
                        seg, corridor, builder,
                        f'corridor_{ci}_sta_{si}', station_color)
                else:
                    style = _resolve_style(
                        seg, corridor, builder,
                        f'corridor_{ci}_gap_{si}', gap_color,
                        dash=gap_dash)

                path = MapPath(points, style)
                parts.append(path.drawSpline(adds=f'class="{style.name}"'))

                # Pawn markers at station hexes
                if show_pawns and seg.type == 'station':
                    for ph in seg.hexes:
                        if 0 <= ph < len(grid.hexes):
                            c = grid.hexes[ph].center
                            parts.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" '
                                f'r="{pawn_radius}" class="{style.name}"/>')

            # Target glyph at corridor endpoint
            if show_targets and hasattr(corridor, 'target') and corridor.target:
                tgt = corridor.target
                th = tgt.hex
                if 0 <= th < len(grid.hexes):
                    tc = grid.hexes[th].center
                    if tgt.glyph:
                        parts.append(tgt.glyph(tc))
                    elif tgt.style:
                        builder.add_style(tgt.style)
                        parts.append(
                            f'<circle cx="{tc.x:.1f}" cy="{tc.y:.1f}" '
                            f'r="{target_radius}" class="{tgt.style.name}"/>')

        return '\n'.join(parts)

    return OverlaySpec('corridors', render, requires=set(), priority=55)
```

Key design choices:

- **Style resolution**: segment → corridor → generated default (3-level fallback)
- **Priority 55**: sits between climate (40) and rivers (60), so corridors draw over terrain but under rivers
- **No hard `requires`**: checks `ctx.corridors` internally, returns `''` if absent — same pattern as your `RiverOverlay`
- **`target.glyph`**: callable `(center) → svg_str`, so you can plug in stars, diamonds, whatever

Does the priority ordering and the glyph-as-callable approach feel right?

In [ ]:
@patch
def badge(self: DiagramGlyphs, center: MapCord, shape: str = 'shield',
          size: float = None, fill: str = None, stroke: str = None,
          flag: 'CountryFlag' = None, builder: 'SVGBuilder' = None) -> str:
    """Heraldic badge shape at center, themed from flag.
    
    Args:
        flag:    optional CountryFlag — uses its pattern fill
        builder: required if flag is passed (to register the pattern def)
    """
    s = size or self.size * 1.5
    stroke = stroke or self.color

    shapes = {'shield': BADGE_SHIELD, 'banner': BADGE_BANNER}
    path_svg = shapes.get(shape, BADGE_SHIELD)

    # Resolve fill: flag pattern → explicit fill → glyph default
    if flag and builder:
        norm = s / 500
        pat_id = f"badge_{flag.name}_{id(center)}"
        badge_fill, pat_def = flag.piece_fill(pat_id, size='large', scale=norm)
        if pat_def:
            builder.add_definition(pat_def)
    elif fill:
        badge_fill = fill
    else:
        badge_fill = self.fill

    norm = s / 500
    tx, ty = center.x, center.y

    return (f'<g transform="translate({tx},{ty}) scale({norm:.4f}) translate(-500,-500)">'
            f'<g fill="{badge_fill}" stroke="{stroke}" stroke-width="{max(8, 20/norm):.0f}">'
            f'{path_svg}</g></g>\n')


In [ ]:
@patch
def target_glyph(self: DiagramGlyphs, target: 'Target',
                 center: MapCord, badge_shape: str = 'shield') -> str:
    """Render a target marker: badge backdrop + purpose icon."""
    svg = self.badge(center, shape=badge_shape)

    dispatch = {
        'settle':  self.house,
        'harvest': self.sheaf,
        'attack':  self.shield,
        'explore': self.ring,
    }
    fn = dispatch.get(target.purpose, self.ring)

    # Inner glyph sits centered, slightly smaller
    inner_size = self.size * 0.6
    svg += fn(center, size=inner_size)
    return svg


In [ ]:
def CorridorOverlay(
    default_color='#2c3e50',
    gap_color='#e74c3c',
    gap_dash='8,4',
    station_color='#27ae60',
    pawn_radius=4,
    path_width=3.0,
    show_pawns=True,
    show_targets=True,
    badge_shape='shield',
    default_glyph: DiagramGlyphs = None,
    **kw
) -> OverlaySpec:
    """Render corridors with station/caravan segments, pawn markers, and target badges.

    Pulls corridor list from ctx.corridors (flat list).
    Style resolution: segment.style → corridor.style → overlay defaults.
    Glyph resolution: corridor.glyph → default_glyph → plain circles.
    """

    def _resolve_style(seg, corridor, builder, default_name, default_fill,
                       dash=None, width=path_width):
        """Segment style → corridor style → generated default."""
        style = seg.style or corridor.style
        if style:
            builder.add_style(style)
            return style
        s = StyleCSS(default_name, fill='none', stroke=default_fill,
                     stroke_width=width, opacity=0.85,
                     stroke_linecap='round', stroke_linejoin='round')
        if dash:
            s.properties['stroke-dasharray'] = dash
        builder.add_style(s)
        return s

    def _resolve_glyph(corridor):
        return corridor.glyph or default_glyph

    def render(ctx: OverlayContext) -> str:
        corridors = getattr(ctx, 'corridors', None)
        if not corridors:
            return ''

        grid = ctx.grid
        builder = ctx.builder
        parts = []

        for ci, corridor in enumerate(corridors):
            route = corridor.hexes
            if not route:
                continue

            glyph = _resolve_glyph(corridor)
            if glyph:
                glyph.register_styles(builder)

            segs = corridor.segments or []

            # ── No segments: draw whole route as single path ──
            if not segs:
                style = corridor.style
                if not style:
                    style = StyleCSS(f'corridor_{ci}', fill='none',
                                     stroke=default_color, stroke_width=path_width,
                                     opacity=0.85, stroke_linecap='round')
                builder.add_style(style)
                points = [grid.hexes[h].center for h in route
                          if 0 <= h < len(grid.hexes)]
                if len(points) >= 2:
                    parts.append(MapPath(points, style).drawSpline(
                        adds=f'class="{style.name}"'))
                continue

            # ── Draw each segment ──
            for si, seg in enumerate(segs):
                seg_hexes = route[seg.route_start:seg.route_end + 1]
                points = [grid.hexes[h].center for h in seg_hexes
                          if 0 <= h < len(grid.hexes)]
                if len(points) < 2:
                    continue

                is_station = seg.type == 'station'

                if is_station:
                    style = _resolve_style(seg, corridor, builder,
                                           f'cor_{ci}_sta_{si}', station_color)
                else:
                    style = _resolve_style(seg, corridor, builder,
                                           f'cor_{ci}_gap_{si}', gap_color,
                                           dash=gap_dash)

                parts.append(MapPath(points, style).drawSpline(
                    adds=f'class="{style.name}"'))

                # ── Pawn markers at station hexes ──
                if show_pawns and is_station:
                    for pi, ph in enumerate(seg.hexes):
                        if ph < 0 or ph >= len(grid.hexes):
                            continue
                        c = grid.hexes[ph].center
                        if glyph and seg.exports:
                            # Food pip sized by export fraction
                            max_exp = max(seg.exports) if seg.exports else 1
                            frac = seg.exports[pi] / max_exp if pi < len(seg.exports) else 0.5
                            parts.append(glyph.food_pip(c, frac, max_r=pawn_radius * 1.5))
                        elif glyph:
                            parts.append(glyph.circle(c, pawn_radius,
                                                      glyph.filled_style()))
                        else:
                            parts.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" '
                                f'r="{pawn_radius}" fill="{station_color}" '
                                f'opacity="0.85"/>')

            # ── Target badge at corridor endpoint ──
            if show_targets and corridor.target:
                tgt = corridor.target
                if 0 <= tgt.hex < len(grid.hexes):
                    tc = grid.hexes[tgt.hex].center

                    if glyph:
                        parts.append(glyph.target_glyph(
                            tgt, tc, badge_shape=badge_shape))
                    elif tgt.style:
                        builder.add_style(tgt.style)
                        parts.append(
                            f'<circle cx="{tc.x:.1f}" cy="{tc.y:.1f}" '
                            f'r="8" class="{tgt.style.name}"/>')

        return '\n'.join(parts)

    return OverlaySpec('corridors', render, requires=set(), priority=55)


So we now have a game context should/can we rewrite CorridorOverlay. We should also consider about using DiagramGlyphs and creating ones we would need

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call. Let's think about what we need from `DiagramGlyphs` for corridors:

**Existing glyphs that map well:**
- `sheaf` → harvest targets / food stations
- `house` → settle targets
- `shield` → attack targets
- `ring` → explore targets / generic waypoints
- `transfer_arrow` → food flow between stations
- `food_pip` → per-station surplus indicator

**New glyphs we'd want:**
- **`camp`** — tent/campfire shape for pawn stations (distinct from `house` which is settlements)
- **`target_marker`** — dispatches by `Target.purpose` to the right glyph automatically

One design question: `DiagramGlyphs` is created from a `CountryFlag`, so it carries kingdom-themed colors. Since corridors from different kingdoms have different flags, should the overlay:

**A)** Accept a single default `DiagramGlyphs` and use it for all corridors
**B)** Derive a `DiagramGlyphs` per corridor from its flag (corridor would need to carry or reference a flag)
**C)** Let corridors optionally carry their own `DiagramGlyphs`, fallback to a default

I'd lean **C** — it's the same fallback pattern as styles. We could add an optional `glyph: DiagramGlyphs = None` field to `Corridor`.

Does that feel right, or would you rather keep glyphs out of the corridor object and handle it differently?

do you want to rewrite cooridor so that it has DiagramGlyphs. do we just add a property?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, just add an optional field — same pattern as `style` and `target`:

```python
@dataclass
class Corridor:
    hexes:           list[int]        = field(default_factory=list)
    kind:            CorridorType     = CorridorType.PERMANENT
    segments:        list[Segment]    = field(default_factory=list)
    n_stations:      int              = 0
    n_gaps:          int              = 0
    fuel_needed:     float            = 0.0
    bishop_min_fuel: float            = 0.0
    duration:        int              = 0
    congestion:      int              = 0
    valid:           bool             = True
    style:           StyleCSS         = None
    target:          Target           = None
    glyph:           DiagramGlyphs    = None
```

Then the overlay resolves like everything else: `corridor.glyph or default_glyph`. When a kingdom creates a corridor it stamps `glyph=DiagramGlyphs(flag=kingdom.flag, size=grid.radius)` and all the markers come out themed automatically.

Anything else we should think about before doing overlay

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

A few things worth settling:

1. **Direction arrows** — Queens travel start→end. A small arrowhead at the corridor endpoint (or periodically along it) would make direction obvious. `DiagramGlyphs` already has `transfer_arrow` — we could reuse that, or just put an arrowhead on the final spline segment.

2. **Target glyph dispatch** — `Target.purpose` is a string (`'attack'`, `'settle'`, `'harvest'`, `'explore'`). Rather than leaving it to the caller to set `target.glyph`, we could add a method to `DiagramGlyphs`:

   ```python
   def target_glyph(self, purpose: str, center: MapCord) -> str:
       dispatch = {
           'settle':  self.house,
           'harvest': self.sheaf,
           'attack':  self.shield,
           'explore': self.ring,
       }
       fn = dispatch.get(purpose, self.ring)
       return fn(center)
   ```

3. **Camp glyph** — We said pawn stations need something distinct from `house` (which is settlements). A small tent/campfire shape. Worth adding to `DiagramGlyphs` before the overlay, or just use filled circles for v1?

4. **Segment transitions** — Where a station meets a caravan, a small junction dot might help visually. Or skip for v1.

5. **`requires` set** — Since `GameContext` has `ctx.corridors`, should the overlay declare `requires={'corridors'}` or keep the soft check? Your other overlays (rivers, settlements) do soft checks, so probably stay consistent.

I'd say items 1 and 2 are worth doing now, 3 and 4 can wait. Sound right?

In [ ]:
def CorridorOverlay(
    default_color='#2c3e50',
    gap_color='#e74c3c',
    gap_dash='8,4',
    station_color='#27ae60',
    pawn_radius=4,
    path_width=3.0,
    show_pawns=True,
    show_targets=True,
    badge_shape='shield',
    default_glyph: DiagramGlyphs = None,
    **kw
) -> OverlaySpec:
    """Render corridors with station/caravan segments, pawn markers, and target badges.

    Pulls corridor list from ctx.corridors (flat list).
    Style resolution: segment.style → corridor.style → overlay defaults.
    Glyph resolution: corridor.glyph → default_glyph → plain circles.
    """

    def _resolve_style(seg, corridor, builder, default_name, default_fill,
                       dash=None, width=path_width):
        """Segment style → corridor style → generated default."""
        style = seg.style or corridor.style
        if style:
            builder.add_style(style)
            return style
        s = StyleCSS(default_name, fill='none', stroke=default_fill,
                     stroke_width=width, opacity=0.85,
                     stroke_linecap='round', stroke_linejoin='round')
        if dash:
            s.properties['stroke-dasharray'] = dash
        builder.add_style(s)
        return s

    def _resolve_glyph(corridor):
        return corridor.glyph or default_glyph

    def render(ctx: OverlayContext) -> str:
        corridors = getattr(ctx, 'corridors', None)
        if not corridors:
            return ''

        grid = ctx.grid
        builder = ctx.builder
        parts = []

        for ci, corridor in enumerate(corridors):
            route = corridor.hexes
            if not route:
                continue

            glyph = _resolve_glyph(corridor)
            if glyph:
                glyph.register_styles(builder)

            segs = corridor.segments or []

            # ── No segments: draw whole route as single path ──
            if not segs:
                style = corridor.style
                if not style:
                    style = StyleCSS(f'corridor_{ci}', fill='none',
                                     stroke=default_color, stroke_width=path_width,
                                     opacity=0.85, stroke_linecap='round')
                builder.add_style(style)
                points = [grid.hexes[h].center for h in route
                          if 0 <= h < len(grid.hexes)]
                if len(points) >= 2:
                    parts.append(MapPath(points, style).drawSpline(
                        adds=f'class="{style.name}"'))
                continue

            # ── Draw each segment ──
            for si, seg in enumerate(segs):
                seg_hexes = route[seg.route_start:seg.route_end + 1]
                points = [grid.hexes[h].center for h in seg_hexes
                          if 0 <= h < len(grid.hexes)]
                if len(points) < 2:
                    continue

                is_station = seg.type == 'station'

                if is_station:
                    style = _resolve_style(seg, corridor, builder,
                                           f'cor_{ci}_sta_{si}', station_color)
                else:
                    style = _resolve_style(seg, corridor, builder,
                                           f'cor_{ci}_gap_{si}', gap_color,
                                           dash=gap_dash)

                parts.append(MapPath(points, style).drawSpline(
                    adds=f'class="{style.name}"'))

                # ── Pawn markers at station hexes ──
                if show_pawns and is_station:
                    for pi, ph in enumerate(seg.hexes):
                        if ph < 0 or ph >= len(grid.hexes):
                            continue
                        c = grid.hexes[ph].center
                        if glyph and seg.exports:
                            # Food pip sized by export fraction
                            max_exp = max(seg.exports) if seg.exports else 1
                            frac = seg.exports[pi] / max_exp if pi < len(seg.exports) else 0.5
                            parts.append(glyph.food_pip(c, frac, max_r=pawn_radius * 1.5))
                        elif glyph:
                            parts.append(glyph.circle(c, pawn_radius,
                                                      glyph.filled_style()))
                        else:
                            parts.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" '
                                f'r="{pawn_radius}" fill="{station_color}" '
                                f'opacity="0.85"/>')

            # ── Target badge at corridor endpoint ──
            if show_targets and corridor.target:
                tgt = corridor.target
                if 0 <= tgt.hex < len(grid.hexes):
                    tc = grid.hexes[tgt.hex].center

                    if glyph:
                        parts.append(glyph.target_glyph(
                            tgt, tc, badge_shape=badge_shape))
                    elif tgt.style:
                        builder.add_style(tgt.style)
                        parts.append(
                            f'<circle cx="{tc.x:.1f}" cy="{tc.y:.1f}" '
                            f'r="8" class="{tgt.style.name}"/>')

        return '\n'.join(parts)

    return OverlaySpec('corridors', render, requires=set(), priority=55)


In [ ]:
# ── Test route from land hexes ──
grid = myStuff.grid
terr = myStuff.terr
land = [i for i in range(len(terr.elevations))
        if terr.elevations[i] > 0 and i not in grid.invalidRegion]

from collections import deque
def bfs_route(start, grid, elevs, length=15):
    q = deque([(start, [start])])
    vis = {start}
    best = [start]
    while q:
        cur, path = q.popleft()
        if len(path) > len(best): best = path
        if len(path) >= length: return path
        for nb in grid.neighborsOf(cur):
            if nb >= 0 and nb not in vis and nb < len(elevs) and elevs[nb] > 0:
                vis.add(nb)
                q.append((nb, path + [nb]))
    return best

route = bfs_route(land[len(land)//3], grid, terr.elevations, 15)
mid = len(route) // 2
print(f"Test route: {len(route)} hexes, ⬡{route[0]}→⬡{route[-1]}")

# ── Build segments ──
seg1 = Segment(type='station', route_start=0, route_end=mid-1,
               hexes=[route[1], route[mid-2]], tiers=[3, 4],
               exports=[2.0, 2.5], total_export=4.5, n_pawns=2)

seg2 = Segment(type='caravan', route_start=mid, route_end=len(route)-1,
               gap_hexes=route[mid:], gap_len=len(route)-mid,
               load_hex=route[mid-1], dock_hex=route[-1],
               est_turns=3, fuel_needed=31.5, burn_rate=10.5)

# ── Build corridor directly ──
test_corridor = Corridor(
    hexes=route, kind=CorridorType.PERMANENT,
    segments=[seg1, seg2],
    n_stations=1, n_gaps=1,
    fuel_needed=31.5, bishop_min_fuel=140.0,
)


In [ ]:
ctx = myStuff.overlayContext()

TerrainDisplay(
    SettlementOverlay(),
    CorridorOverlay(),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    corridors=[test_corridor],
    context_cls=GameContext,
    debug = not showDemo
)



Should we think about redoing __ft__ now that we have glyphs and styles

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, good time to think about it. The current `__ft__` methods are pure text/tables. Now that we have styles and glyphs, we could add visual cues without going full SVG-in-cards:

**Quick wins:**
- **Segment** — colored dot or bar swatch from `style.stroke` next to "Station" / "Caravan" label (instead of just emoji)
- **Corridor** — small inline SVG showing a colored line with station dots and dashed gaps — a mini schematic
- **Target** — inline the purpose glyph as a tiny SVG badge

**What feels practical:**
- Use `style.properties.get('stroke', default_color)` to pull the color for badges/labels
- For glyphs, wrap a small `<svg viewBox>` inline — `DiagramGlyphs` methods return SVG strings so they'd embed directly

**What might be overkill:**
- Full corridor map rendering in `__ft__` — that's the overlay's job

I'd suggest: **Segment and Corridor get color swatches from their styles, Target gets a tiny glyph badge.** Something like:

```python
def _color_dot(color, size=12):
    """Inline SVG color indicator."""
    return NotStr(
        f'<svg width="{size}" height="{size}" style="vertical-align:middle;display:inline-block">'
        f'<circle cx="{size//2}" cy="{size//2}" r="{size//2-1}" fill="{color}"/></svg>')

def _style_color(style, fallback='#888'):
    """Extract stroke or fill color from a StyleCSS."""
    if not style: return fallback
    return style.properties.get('stroke', style.properties.get('fill', fallback))
```

Then in `Segment.__ft__`, replace the emoji with:

```python
dot = _color_dot(_style_color(self.style, '#27ae60' if self.type == 'station' else '#e74c3c'))
```

And for `Corridor.__ft__`, a mini schematic — just a horizontal line with dots:

```python
def _corridor_schematic(corridor, width=200, height=24):
    """Tiny inline SVG showing station/gap pattern."""
    if not corridor.segments or not corridor.hexes: return ''
    total = len(corridor.hexes)
    parts = []
    for seg in corridor.segments:
        x1 = seg.route_start / total * width
        x2 = (seg.route_end + 1) / total * width
        color = _style_color(seg.style, '#27ae60' if seg.type == 'station' else '#e74c3c')
        dash = ' stroke-dasharray="4,3"' if seg.type == 'caravan' else ''
        parts.append(f'<line x1="{x1:.0f}" y1="12" x2="{x2:.0f}" y2="12" '
                     f'stroke="{color}" stroke-width="3"{dash}/>')
    svg = (f'<svg width="{width}" height="{height}" '
           f'style="vertical-align:middle;display:inline-block">{"".join(parts)}</svg>')
    return NotStr(svg)
```

Want to go with this lightweight approach, or push deeper into embedding the glyph shapes?

I think lightweight would be great.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
def _color_dot(color, size=12):
    """Inline SVG color indicator."""
    return NotStr(
        f'<svg width="{size}" height="{size}" style="vertical-align:middle;display:inline-block">'
        f'<circle cx="{size//2}" cy="{size//2}" r="{size//2-1}" fill="{color}"/></svg>')

def _style_color(style, fallback='#888'):
    """Extract stroke or fill color from a StyleCSS."""
    if not style: return fallback
    return style.properties.get('stroke', style.properties.get('fill', fallback))

def _corridor_schematic(corridor, width=200, height=24):
    """Tiny inline SVG showing station/gap pattern."""
    if not corridor.segments or not corridor.hexes: return ''
    total = len(corridor.hexes) or 1
    parts = []
    for seg in corridor.segments:
        x1 = seg.route_start / total * width
        x2 = (seg.route_end + 1) / total * width
        color = _style_color(seg.style, '#27ae60' if seg.type == 'station' else '#e74c3c')
        dash = ' stroke-dasharray="4,3"' if seg.type == 'caravan' else ''
        parts.append(f'<line x1="{x1:.0f}" y1="12" x2="{x2:.0f}" y2="12" '
                     f'stroke="{color}" stroke-width="3" stroke-linecap="round"{dash}/>')
        # Station pawn dots
        if seg.type == 'station' and seg.hexes:
            for ph in seg.hexes:
                px = (seg.route_start + (seg.route_end - seg.route_start) / 2) / total * width
                parts.append(f'<circle cx="{px:.0f}" cy="12" r="3" fill="{color}"/>')
    svg = (f'<svg width="{width}" height="{height}" '
           f'style="vertical-align:middle;display:inline-block">{"".join(parts)}</svg>')
    return NotStr(svg)
```

```python
@patch
def __ft__(self: Segment):
    is_station = self.type == 'station'
    color = _style_color(self.style, '#27ae60' if is_station else '#e74c3c')
    dot = _color_dot(color)
    label = 'Station' if is_station else 'Caravan'
    span_txt = f"route[{self.route_start}..{self.route_end}] ({self.span} hex)"

    if is_station:
        rows = []
        for h, t, e in zip(self.hexes, self.tiers, self.exports):
            rows.append(Tr(Td(f'⬡{h}'), Td(f'T{t}'), Td(f'+{e:.1f}/t')))
        body = Div(
            P(Small(span_txt), cls='uk-text-muted'),
            Table(
                Thead(Tr(Th('Hex'), Th('Tier'), Th('Export'))),
                Tbody(*rows),
                cls=(TableT.sm, TableT.divider),
            ) if rows else None,
            P(Strong(f'Total: {self.total_export:.1f}/t'),
              Small(f' ({self.n_pawns} pawns)'), cls='uk-text-right'),
        )
    else:
        body = Div(
            P(Small(span_txt), cls='uk-text-muted'),
            Dl(
                Dt('Gap'),       Dd(f'{self.gap_len} hexes'),
                Dt('Load→Dock'), Dd(f'⬡{self.load_hex} → ⬡{self.dock_hex}'),
                Dt('Fuel'),      Dd(f'{self.fuel_needed:.1f} ({self.burn_rate:.1f}/t × ~{self.est_turns}t)'),
                cls='uk-description-list',
            ),
        )

    return Card(
        CardHeader(CardTitle(Span(dot, f' {label}'))),
        body,
        cls='uk-card-small',
    )
```

```python
@patch
def __ft__(self: Corridor):
    status = '✅' if self.feasible else '❌'
    kind_badge = 'primary' if self.kind == CorridorType.PERMANENT else 'muted'
    color = _style_color(self.style, '#2c3e50')

    header = Div(
        _color_dot(color, 14),
        Span(f' ⬡{self.start} → ⬡{self.end}', cls='uk-text-bold'),
        Span(f' ({self.length} hex)', cls='uk-text-muted'),
        Span(f' {self.kind.value}', cls=f'uk-label uk-label-{kind_badge} uk-margin-small-left'),
        Span(f' {status}', cls='uk-margin-small-left'),
    )

    schematic = _corridor_schematic(self)

    stats = Div(
        Small(f'{self.n_stations} stations · {self.n_gaps} gaps'),
        Small(f' · fuel {self.fuel_needed:.0f}/{self.bishop_min_fuel:.0f}') if self.n_gaps else None,
        cls='uk-text-muted',
    )

    seg_cards = Div(*[s.__ft__() for s in self.segments],
                    cls='uk-grid uk-child-width-1-2@m uk-grid-small') if self.segments else P(
        Em('No segments'), cls='uk-text-muted')

    return Card(header, schematic, stats, seg_cards, cls='uk-card-small')
```

```python
@patch
def __ft__(self: Target):
    color = _style_color(self.style, '#555')
    dl = f'{self.deadline}t' if self.deadline else '—'
    return Tr(
        Td(_color_dot(color, 10), f' ⬡{self.hex}'),
        Td(self.purpose),
        Td(f'{self.priority:.1f}'),
        Td(dl),
        Td(f'{self.payoff:.1f}'),
    )
```

Key changes:

- **Color dots** pulled from `style.stroke` everywhere, with sensible defaults (green for stations, red for caravans, dark for corridors)
- **Schematic** on `Corridor.__ft__` — mini horizontal line diagram showing station vs gap stretches at a glance
- **No heavy SVG** — just tiny inline circles and lines, renders fine in any HTML context

In [ ]:
def _color_dot(color, size=12):
    """Inline SVG color indicator."""
    return NotStr(
        f'<svg width="{size}" height="{size}" style="vertical-align:middle;display:inline-block">'
        f'<circle cx="{size//2}" cy="{size//2}" r="{size//2-1}" fill="{color}"/></svg>')

def _style_color(style, fallback='#888'):
    """Extract stroke or fill color from a StyleCSS."""
    if not style: return fallback
    return style.properties.get('stroke', style.properties.get('fill', fallback))

def _corridor_schematic(corridor, width=200, height=24):
    """Tiny inline SVG showing station/gap pattern."""
    if not corridor.segments or not corridor.hexes: return ''
    total = len(corridor.hexes) or 1
    parts = []
    for seg in corridor.segments:
        x1 = seg.route_start / total * width
        x2 = (seg.route_end + 1) / total * width
        color = _style_color(seg.style, '#27ae60' if seg.type == 'station' else '#e74c3c')
        dash = ' stroke-dasharray="4,3"' if seg.type == 'caravan' else ''
        parts.append(f'<line x1="{x1:.0f}" y1="12" x2="{x2:.0f}" y2="12" '
                     f'stroke="{color}" stroke-width="3" stroke-linecap="round"{dash}/>')
        # Station pawn dots
        if seg.type == 'station' and seg.hexes:
            for ph in seg.hexes:
                px = (seg.route_start + (seg.route_end - seg.route_start) / 2) / total * width
                parts.append(f'<circle cx="{px:.0f}" cy="12" r="3" fill="{color}"/>')
    svg = (f'<svg width="{width}" height="{height}" '
           f'style="vertical-align:middle;display:inline-block">{"".join(parts)}</svg>')
    return NotStr(svg)


In [ ]:
@patch
def __ft__(self: Segment):
    is_station = self.type == 'station'
    color = _style_color(self.style, '#27ae60' if is_station else '#e74c3c')
    dot = _color_dot(color)
    label = 'Station' if is_station else 'Caravan'
    span_txt = f"route[{self.route_start}..{self.route_end}] ({self.span} hex)"

    if is_station:
        rows = []
        for h, t, e in zip(self.hexes, self.tiers, self.exports):
            rows.append(Tr(Td(f'⬡{h}'), Td(f'T{t}'), Td(f'+{e:.1f}/t')))
        body = Div(
            P(Small(span_txt), cls='uk-text-muted'),
            Table(
                Thead(Tr(Th('Hex'), Th('Tier'), Th('Export'))),
                Tbody(*rows),
                cls=(TableT.sm, TableT.divider),
            ) if rows else None,
            P(Strong(f'Total: {self.total_export:.1f}/t'),
              Small(f' ({self.n_pawns} pawns)'), cls='uk-text-right'),
        )
    else:
        body = Div(
            P(Small(span_txt), cls='uk-text-muted'),
            Dl(
                Dt('Gap'),       Dd(f'{self.gap_len} hexes'),
                Dt('Load→Dock'), Dd(f'⬡{self.load_hex} → ⬡{self.dock_hex}'),
                Dt('Fuel'),      Dd(f'{self.fuel_needed:.1f} ({self.burn_rate:.1f}/t × ~{self.est_turns}t)'),
                cls='uk-description-list',
            ),
        )

    return Card(
        CardHeader(CardTitle(Span(dot, f' {label}'))),
        body,
        cls='uk-card-small',
    )


In [ ]:
@patch
def __ft__(self: Target):
    color = _style_color(self.style, '#555')
    dl = f'{self.deadline}t' if self.deadline else '—'
    return Tr(
        Td(_color_dot(color, 10), f' ⬡{self.hex}'),
        Td(self.purpose),
        Td(f'{self.priority:.1f}'),
        Td(dl),
        Td(f'{self.payoff:.1f}'),
    )


In [ ]:



@patch
def __ft__(self: Corridor):
    status = '✅' if self.feasible else '❌'
    kind_badge = 'primary' if self.kind == CorridorType.PERMANENT else 'muted'
    color = _style_color(self.style, '#2c3e50')

    header = Div(
        _color_dot(color, 14),
        Span(f' ⬡{self.start} → ⬡{self.end}', cls='uk-text-bold'),
        Span(f' ({self.length}hex)', cls='uk-text-muted'),
        Span(f' {self.kind.value}', cls=f'uk-label uk-label-{kind_badge} uk-margin-small-left'),
        Span(f' {status}'),
    )

    schematic = _corridor_schematic(self)

    stats = Small(
        f'{self.n_stations} stations · {self.n_gaps} gaps'
        + (f' · fuel {self.fuel_needed:.0f}/{self.bishop_min_fuel:.0f}' if self.n_gaps else ''),
        cls='uk-text-muted',
    )

    if self.segments:
        seg_row = Div(*[s.__ft__() for s in self.segments],
                      cls='uk-flex uk-flex-row uk-flex-wrap',
                      style='gap:0.5rem')
    else:
        seg_row = P(Em('No segments'), cls='uk-text-muted')

    return Div(Card(header, schematic, stats, cls='uk-card-small'),seg_row)


In [ ]:
@patch
def __ft__(self: Corridor):
    status = '✅' if self.feasible else '❌'
    kind_badge = 'primary' if self.kind == CorridorType.PERMANENT else 'muted'
    color = _style_color(self.style, '#2c3e50')

    header = CardHeader(CardTitle(
        _color_dot(color, 14),
        f' ⬡{self.start} → ⬡{self.end}',
        Span(f' ({self.length}hex)', cls='uk-text-muted uk-text-small'),
        Span(f' {self.kind.value}', cls=f'uk-label uk-label-{kind_badge} uk-margin-small-left'),
        Span(f' {status}'),
    ))

    schematic = _corridor_schematic(self)

    stats = P(
        Small(f'{self.n_stations} stations · {self.n_gaps} gaps'
              + (f' · fuel {self.fuel_needed:.0f}/{self.bishop_min_fuel:.0f}' if self.n_gaps else '')),
        cls='uk-text-muted',
    )

    if self.segments:
        seg_row = Div(*[s.__ft__() for s in self.segments],
                      cls='uk-grid uk-child-width-1-2@s uk-grid-small',
                      **{'uk-grid': True})
    else:
        seg_row = P(Em('No segments'), cls='uk-text-muted')

    return Card(
        header,
        Div(schematic, stats, cls='uk-margin-small'),
        seg_row,
        cls='uk-card-small uk-card-default',
    )


can we test these new __ft__

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
# Test Segment __ft__
print("── Station Segment ──")
seg1

```

```python
# Test Caravan Segment
print("── Caravan Segment ──")
seg2
```

```python
# Test Corridor __ft__
print("── Corridor ──")
test_corridor
```

```python
# Test Target __ft__
tgt = Target(hex=route[-1], purpose='settle', priority=2.0, deadline=10, payoff=15.0)
print("── Target (in a table) ──")
Table(
    Thead(Tr(Th('Hex'), Th('Purpose'), Th('Priority'), Th('Deadline'), Th('Payoff'))),
    Tbody(tgt),
    cls=TableT.sm,
)
```

That should show each one with the color dots and schematic. Try running those and let's see how they look!

In [ ]:
show(seg1)

In [ ]:
show(seg2)

In [ ]:
show(test_corridor)

## Sim Adapter

In [ ]:
# ── Schedule schema + helpers ─────────────────────────────────────────────

SCHEDULE_COLUMNS = [
    'schedule_id', 'piece_id', 'piece_name', 'turn', 'hex', 'facing',
    'action', 'food', 'health', 'segment_id', 'segment_type',
    'corridor_id', 'note',
]

def _exact_facing(grid, from_hex, to_hex):
    "Exact facing from from_hex to adjacent to_hex, or None."
    dirs = HexPosition.directions()
    for d in range(6):
        if grid.hexposition_to_index(dirs[d], from_hex) == to_hex:
            return d
    return None

def empty_schedule(schedule_id='empty'):
    return pd.DataFrame(columns=SCHEDULE_COLUMNS)

def schedule_timetable(df, title='Schedule', max_cols=25):
    "Render a Schedule DataFrame as a pieces×turns timetable."
    if df.empty: return Card(P('Empty schedule'), header=CardTitle(title))

    pieces = df.piece_name.unique()
    turns  = sorted(df.turn.unique())
    if len(turns) > max_cols:
        step = max(1, len(turns) // max_cols)
        turns = turns[::step]

    header = Tr(Th('piece'), *[Th(f't{int(t)}') for t in turns])
    rows = []
    for pname in pieces:
        pdata = df[df.piece_name == pname].set_index('turn')
        cells = [Th(pname)]
        for t in turns:
            if t in pdata.index:
                r = pdata.loc[t]
                if isinstance(r, pd.DataFrame): r = r.iloc[0]
                h = int(r['hex']); f = int(r['facing']); fd = r['food']
                tip = f"f={f} food={fd:.1f} {r.get('action','—')}"
                cells.append(Td(f'⬡{h}', title=tip, style='font-size:0.8em'))
            else:
                cells.append(Td('—'))
        rows.append(Tr(*cells))

    return Card(
        Table(Thead(header), Tbody(*rows), cls=TableT.sm),
        header=CardTitle(title),
    )


In [ ]:
_EVENT_TO_ACTION = {
    'moved': 'FORWARD', 'bumped': 'BUMPED', 'harvested': 'HARVEST',
    'gave': 'GIVE', 'rotated': 'ROT', 'blocked': 'BLOCKED',
    'attacked': 'ATTACK', 'countered': 'COUNTER',
}
_ACTION_RANK = {
    'FORWARD': 0, 'BUMPED': 1, 'ATTACK': 2, 'COUNTER': 3,
    'HARVEST': 4, 'GIVE': 5, 'BLOCKED': 6, 'ROT': 7,
}

@dataclass
class SimAdapter:
    """Extracts actual schedules from FoodSimulator output.

    Sim snapshots (post-step state) are offset +1 so turn 0 = pre-sim.
    """
    sim:       FoodSimulator
    corridors: list[Corridor]
    _names:    dict = field(default_factory=dict)

    def __post_init__(self):
        self._names = {p.id: p.name for p in self.sim.pieces}

    def _primary_action(self, piece_id, sim_turn):
        "Most significant voluntary action for (piece, sim_turn)."
        df = self.sim.df
        evts = df.loc[(df.piece_id == piece_id) & (df.turn == sim_turn), 'event']
        best, best_r = '—', 999
        for ev in evts:
            act = _EVENT_TO_ACTION.get(ev)
            if act and _ACTION_RANK.get(act, 99) < best_r:
                best, best_r = act, _ACTION_RANK[act]
        return best

    def _map_loc(self, hex_idx):
        "Map hex → (corridor_label, segment_label, segment_type) or Nones."
        for cor in self.corridors:
            if hex_idx not in cor: continue
            w = cor.index(hex_idx)
            try:
                seg = cor[w]
                return cor.label, seg.label, seg.type
            except KeyError:
                return cor.label, None, None
        return None, None, None

    def _at_target(self, hex_idx):
        "Check if hex is a corridor target — returns purpose string or None."
        for cor in self.corridors:
            if cor.target and cor.target.hex == hex_idx:
                return cor.target.purpose
        return None

    def _has_event(self, pid, sim_turn, ename):
        df = self.sim.df
        return ((df.piece_id == pid) & (df.turn == sim_turn) & (df.event == ename)).any()

    def _row(self, pid, turn, hex_idx, facing, action, food, health,
             schedule_id, note=''):
        cid, slabel, stype = self._map_loc(hex_idx)
        tgt = self._at_target(hex_idx)
        if tgt and not note:
            note = f'🎯 {tgt}'
        return dict(
            schedule_id=schedule_id, piece_id=pid,
            piece_name=self._names.get(pid, pid[:8]),
            turn=turn, hex=hex_idx, facing=facing,
            action=action, food=round(food, 1),
            health=round(health, 1),
            segment_id=slabel, segment_type=stype,
            corridor_id=cid, note=note)

    def extract(self, schedule_id='actual', initial_states=None, piece_ids=None):
        """Build Schedule DataFrame."""
        ids  = piece_ids or set(self._names)
        rows = []

        # Turn 0 from initial_states
        if initial_states:
            for pid, st in initial_states.items():
                if pid not in ids: continue
                rows.append(self._row(
                    pid, 0, st['location'], st['facing'],
                    '—', st['food'], st.get('health', 100),
                    schedule_id, 'start'))

        # Snapshots → turn 1, 2, ...
        for snap in self.sim.snapshots:
            t = snap['turn'] + 1
            for pid, state in snap['piece_states'].items():
                if pid not in ids: continue
                act = self._primary_action(pid, snap['turn'])
                note = ''
                if self._has_event(pid, snap['turn'], 'died'):      note = '💀 dead'
                elif self._has_event(pid, snap['turn'], 'starved'): note = 'starving'
                rows.append(self._row(
                    pid, t, state['location'], state['facing'],
                    act, state['food'], state['health'],
                    schedule_id, note))

        return pd.DataFrame(rows, columns=SCHEDULE_COLUMNS) if rows else empty_schedule()



In [ ]:
@dataclass
class VarianceReport:
    "Joins budget + actual schedules and flags divergence."
    budget: pd.DataFrame
    actual: pd.DataFrame

    def compute(self):
        v = pd.merge(self.budget, self.actual,
                     on=['piece_id', 'turn'], suffixes=('_bgt', '_act'),
                     how='outer')
        v['hex_match']    = v.hex_bgt == v.hex_act
        v['facing_match'] = v.facing_bgt == v.facing_act
        v['food_delta']   = v.food_act - v.food_bgt

        def _status(r):
            if pd.notna(r.get('note_act')) and '💀' in str(r.note_act): return '💀 dead'
            if pd.isna(r.get('hex_act')): return 'not_started'
            if pd.isna(r.get('hex_bgt')): return 'unplanned'
            if r.hex_match: return 'on_track'
            return 'off_route'

        v['status'] = v.apply(_status, axis=1)
        return v

    def summary(self):
        v  = self.compute()
        nm = v.piece_name_bgt.fillna(v.piece_name_act)
        out = []
        for pname in nm.unique():
            pv = v[nm == pname]
            on = (pv.status == 'on_track').sum()
            fd = pv.food_delta.mean()
            worst = 'on_track'
            for s in ['💀 dead', 'off_route', 'not_started']:
                if (pv.status == s).any(): worst = s; break
            out.append(dict(piece=pname, on_track=f'{on}/{len(pv)}',
                            avg_food_delta=f'{fd:+.1f}' if pd.notna(fd) else '—',
                            status=worst))
        return pd.DataFrame(out)

    def __ft__(self):
        v  = self.compute()
        nm = v.piece_name_bgt.fillna(v.piece_name_act)
        pieces = nm.unique()
        turns  = sorted(v.turn.unique())

        _col = {'on_track': '#d4edda', 'off_route': '#f8d7da',
                '💀 dead': '#555', 'unplanned': '#fff3cd',
                'not_started': '#e2e3e5'}

        header = Tr(Th('piece'), *[Th(f't{int(t)}') for t in turns])
        trows  = []
        for pname in pieces:
            pv = v[nm == pname].set_index('turn')
            cells = [Th(pname)]
            for t in turns:
                if t in pv.index:
                    r = pv.loc[t]
                    if isinstance(r, pd.DataFrame): r = r.iloc[0]
                    st  = r['status']
                    bg  = _col.get(st, '#fff')
                    ha  = r.get('hex_act', r.get('hex_bgt'))
                    hb  = r.get('hex_bgt')
                    hs  = f'⬡{int(ha)}' if pd.notna(ha) else '—'
                    fd  = r.get('food_delta', 0)
                    tip = (f"bgt=⬡{int(hb) if pd.notna(hb) else '?'} "
                           f"{st} food:{fd:+.1f}" if pd.notna(fd) else st)
                    cells.append(Td(hs, title=tip,
                                    style=f'background:{bg};font-size:0.8em'))
                else:
                    cells.append(Td('—', style='background:#f5f5f5'))
            trows.append(Tr(*cells))

        timetable = Table(Thead(header), Tbody(*trows), cls=TableT.sm)

        # Summary table
        s = self.summary()
        sh = list(s.columns)
        sr = [Tr(*[Td(str(row[c])) for c in sh]) for _, row in s.iterrows()]
        summ = Table(Thead(Tr(*[Th(h) for h in sh])), Tbody(*sr), cls=TableT.sm)

        return Div(
            Style("td[title]{cursor:help}"),
            Card(timetable, header=CardTitle('Variance Timetable')),
            Card(summ,      header=CardTitle('Summary')),
        )


We need to refactor SimAdapter and its helpers to use segement, target and corridor objects

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call. The main issues are `_route_w_to_seg` accessing segments as dicts, and `_map_loc` could use richer info from the real objects. Here's the refactor:

```python
def _route_w_to_seg(corridor, w):
    "Map route index w → segment index, or None."
    for i, seg in enumerate(corridor.segments):
        if seg.route_start <= w <= seg.route_end:
            return i
    return None

def _hex_to_route_w(corridor, hex_idx):
    "Map hex index → route position, or None."
    for w, h in enumerate(corridor.hexes):
        if h == hex_idx: return w
    return None

def _corridor_label(corridor):
    "Short label for a corridor."
    kind = corridor.kind.value
    return f"{kind}:⬡{corridor.start}→⬡{corridor.end}"

def _segment_label(corridor, seg_idx):
    "Label for a segment within a corridor."
    if seg_idx is None or seg_idx >= len(corridor.segments): return None
    seg = corridor.segments[seg_idx]
    return f"{seg.type}[{seg.route_start}..{seg.route_end}]"
```

```python
_EVENT_TO_ACTION = {
    'moved': 'FORWARD', 'bumped': 'BUMPED', 'harvested': 'HARVEST',
    'gave': 'GIVE', 'rotated': 'ROT', 'blocked': 'BLOCKED',
    'attacked': 'ATTACK', 'countered': 'COUNTER',
}
_ACTION_RANK = {
    'FORWARD': 0, 'BUMPED': 1, 'ATTACK': 2, 'COUNTER': 3,
    'HARVEST': 4, 'GIVE': 5, 'BLOCKED': 6, 'ROT': 7,
}

@dataclass
class SimAdapter:
    """Extracts actual schedules from FoodSimulator output.

    Decoupled from QueenCoverage — takes sim + corridor list.
    Sim snapshots (post-step state) are offset +1 so turn 0 = pre-sim.
    """
    sim:       FoodSimulator
    corridors: list[Corridor]
    _names:    dict = field(default_factory=dict)

    def __post_init__(self):
        self._names = {p.id: p.name for p in self.sim.pieces}

    def _primary_action(self, piece_id, sim_turn):
        "Most significant voluntary action for (piece, sim_turn)."
        df = self.sim.df
        evts = df.loc[(df.piece_id == piece_id) & (df.turn == sim_turn), 'event']
        best, best_r = '—', 999
        for ev in evts:
            act = _EVENT_TO_ACTION.get(ev)
            if act and _ACTION_RANK.get(act, 99) < best_r:
                best, best_r = act, _ACTION_RANK[act]
        return best

    def _map_loc(self, hex_idx):
        "Map a hex → (corridor_label, segment_index, segment_type) or (None, None, None)."
        for cor in self.corridors:
            w = _hex_to_route_w(cor, hex_idx)
            if w is not None:
                sid = _route_w_to_seg(cor, w)
                seg_type = cor.segments[sid].type if sid is not None else None
                return _corridor_label(cor), sid, seg_type
        return None, None, None

    def _at_target(self, hex_idx):
        "Check if hex is a corridor target — returns purpose string or None."
        for cor in self.corridors:
            if cor.target and cor.target.hex == hex_idx:
                return cor.target.purpose
        return None

    def _has_event(self, pid, sim_turn, ename):
        df = self.sim.df
        return ((df.piece_id == pid) & (df.turn == sim_turn) & (df.event == ename)).any()

    def extract(self, schedule_id='actual', initial_states=None, piece_ids=None):
        """Build Schedule DataFrame.

        Args:
            initial_states: dict {piece_id: {location, facing, food, health}}
                            for turn-0 row (pre-simulation).
            piece_ids:      subset of pieces to include (default: all).
        """
        ids  = piece_ids or set(self._names)
        rows = []

        def _row(pid, turn, hex_idx, facing, action, food, health, note=''):
            cid, sid, stype = self._map_loc(hex_idx)
            # Annotate target arrivals
            tgt = self._at_target(hex_idx)
            if tgt and not note:
                note = f'🎯 {tgt}'
            return dict(
                schedule_id=schedule_id, piece_id=pid,
                piece_name=self._names.get(pid, pid[:8]),
                turn=turn, hex=hex_idx, facing=facing,
                action=action, food=round(food, 1),
                health=round(health, 1),
                segment_id=sid, segment_type=stype,
                corridor_id=cid, note=note)

        # ── Turn 0 from initial_states ──
        if initial_states:
            for pid, st in initial_states.items():
                if pid not in ids: continue
                rows.append(_row(
                    pid, 0, st['location'], st['facing'],
                    '—', st['food'], st.get('health', 100), 'start'))

        # ── Snapshots → turn 1, 2, ... ──
        for snap in self.sim.snapshots:
            t = snap['turn'] + 1
            for pid, state in snap['piece_states'].items():
                if pid not in ids: continue
                act = self._primary_action(pid, snap['turn'])
                note = ''
                if self._has_event(pid, snap['turn'], 'died'):      note = '💀 dead'
                elif self._has_event(pid, snap['turn'], 'starved'): note = 'starving'
                rows.append(_row(
                    pid, t, state['location'], state['facing'],
                    act, state['food'], state['health'], note))

        cols = SCHEDULE_COLUMNS + ['segment_type']
        return pd.DataFrame(rows, columns=cols) if rows else empty_schedule()
```

Key changes:

- **`_route_w_to_seg`** — uses `seg.route_start` / `seg.route_end` instead of dict keys
- **`_corridor_label`** — builds label from `corridor.kind.value` + start/end properties
- **`_map_loc`** — now also returns `seg_type` ('station'/'caravan') so the schedule knows which kind of stretch a piece is in
- **`_at_target`** — new helper that checks if a hex matches any corridor's `Target`, annotates rows with 🎯 + purpose
- **`_row` helper** — DRYs up the row-building logic (was duplicated for turn 0 and snapshots)
- **`segment_type` column** added to the DataFrame — useful for filtering/coloring the timetable

The `SCHEDULE_COLUMNS` list would need updating too:

```python
SCHEDULE_COLUMNS = [
    'schedule_id', 'piece_id', 'piece_name', 'turn', 'hex', 'facing',
    'action', 'food', 'health', 'segment_id', 'segment_type',
    'corridor_id', 'note',
]
```

Should some of these like _corridor_label be @patched onto objects

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, definitely. They all take the object as the first argument — that's a textbook sign they should be methods. Here's what I'd patch:

```python
@patch
@property
def label(self: Corridor):
    "Short label for display."
    return f"{self.kind.value}:⬡{self.start}→⬡{self.end}"

@patch
def route_index(self: Corridor, hex_idx):
    "Map hex index → route position, or None."
    for w, h in enumerate(self.hexes):
        if h == hex_idx: return w
    return None

@patch
def segment_at(self: Corridor, w):
    "Map route index w → Segment object, or None."
    for seg in self.segments:
        if seg.route_start <= w <= seg.route_end:
            return seg
    return None

@patch
@property
def label(self: Segment):
    "Short label for display."
    return f"{self.type}[{self.route_start}..{self.route_end}]"
```

Then `SimAdapter._map_loc` cleans up to:

```python
def _map_loc(self, hex_idx):
    for cor in self.corridors:
        w = cor.route_index(hex_idx)
        if w is not None:
            seg = cor.segment_at(w)
            seg_type = seg.type if seg else None
            return cor.label, seg.label if seg else None, seg_type
    return None, None, None
```

Much more readable, and those methods are useful beyond just `SimAdapter` — anywhere you need "which segment is this hex in?" you'd call `corridor.segment_at(w)` instead of reimplementing the loop.

I made some of these patches including make it so you can itterate over corridors. what should be cleaaned up in SimAdapter

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Now that `Corridor` supports `corridor[w]`, `corridor.index(hex)`, `hex in corridor`, `for seg in corridor`, and `.label`, the standalone helpers can go and `SimAdapter` simplifies nicely:

```python
@dataclass
class SimAdapter:
    """Extracts actual schedules from FoodSimulator output.

    Sim snapshots (post-step state) are offset +1 so turn 0 = pre-sim.
    """
    sim:       FoodSimulator
    corridors: list[Corridor]
    _names:    dict = field(default_factory=dict)

    def __post_init__(self):
        self._names = {p.id: p.name for p in self.sim.pieces}

    def _primary_action(self, piece_id, sim_turn):
        "Most significant voluntary action for (piece, sim_turn)."
        df = self.sim.df
        evts = df.loc[(df.piece_id == piece_id) & (df.turn == sim_turn), 'event']
        best, best_r = '—', 999
        for ev in evts:
            act = _EVENT_TO_ACTION.get(ev)
            if act and _ACTION_RANK.get(act, 99) < best_r:
                best, best_r = act, _ACTION_RANK[act]
        return best

    def _map_loc(self, hex_idx):
        "Map hex → (corridor_label, segment_label, segment_type) or Nones."
        for cor in self.corridors:
            if hex_idx not in cor: continue
            w = cor.index(hex_idx)
            try:
                seg = cor[w]
                return cor.label, seg.label, seg.type
            except KeyError:
                return cor.label, None, None
        return None, None, None

    def _at_target(self, hex_idx):
        "Check if hex is a corridor target — returns purpose string or None."
        for cor in self.corridors:
            if cor.target and cor.target.hex == hex_idx:
                return cor.target.purpose
        return None

    def _has_event(self, pid, sim_turn, ename):
        df = self.sim.df
        return ((df.piece_id == pid) & (df.turn == sim_turn) & (df.event == ename)).any()

    def _row(self, pid, turn, hex_idx, facing, action, food, health,
             schedule_id, note=''):
        cid, slabel, stype = self._map_loc(hex_idx)
        tgt = self._at_target(hex_idx)
        if tgt and not note:
            note = f'🎯 {tgt}'
        return dict(
            schedule_id=schedule_id, piece_id=pid,
            piece_name=self._names.get(pid, pid[:8]),
            turn=turn, hex=hex_idx, facing=facing,
            action=action, food=round(food, 1),
            health=round(health, 1),
            segment_id=slabel, segment_type=stype,
            corridor_id=cid, note=note)

    def extract(self, schedule_id='actual', initial_states=None, piece_ids=None):
        """Build Schedule DataFrame."""
        ids  = piece_ids or set(self._names)
        rows = []

        # Turn 0 from initial_states
        if initial_states:
            for pid, st in initial_states.items():
                if pid not in ids: continue
                rows.append(self._row(
                    pid, 0, st['location'], st['facing'],
                    '—', st['food'], st.get('health', 100),
                    schedule_id, 'start'))

        # Snapshots → turn 1, 2, ...
        for snap in self.sim.snapshots:
            t = snap['turn'] + 1
            for pid, state in snap['piece_states'].items():
                if pid not in ids: continue
                act = self._primary_action(pid, snap['turn'])
                note = ''
                if self._has_event(pid, snap['turn'], 'died'):      note = '💀 dead'
                elif self._has_event(pid, snap['turn'], 'starved'): note = 'starving'
                rows.append(self._row(
                    pid, t, state['location'], state['facing'],
                    act, state['food'], state['health'],
                    schedule_id, note))

        return pd.DataFrame(rows, columns=SCHEDULE_COLUMNS) if rows else empty_schedule()
```

What changed:

- **Deleted** `_route_w_to_seg`, `_hex_to_route_w`, `_corridor_label`, `_segment_label` — all replaced by corridor dunders/properties
- **`_map_loc`** — now `hex_idx not in cor` / `cor.index()` / `cor[w]` instead of standalone functions
- **`_row` promoted** to a method — DRYs up the duplicated turn-0 / snapshot row building
- **`segment_id`** now stores `seg.label` (e.g. `"station[0..6]"`) instead of a numeric index — more informative in the DataFrame
- **`SCHEDULE_COLUMNS`** needs updating to include `segment_type`:

```python
SCHEDULE_COLUMNS = [
    'schedule_id', 'piece_id', 'piece_name', 'turn', 'hex', 'facing',
    'action', 'food', 'health', 'segment_id', 'segment_type',
    'corridor_id', 'note',
]
```

Are there any of these helper functions I could take out?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking at the standalone helpers, these four are now fully replaced by corridor dunders/properties and can be deleted:

- **`_route_w_to_seg(corridor, w)`** → `corridor[w]`
- **`_hex_to_route_w(corridor, hex_idx)`** → `corridor.index(hex_idx)`
- **`_corridor_label(corridor)`** → `corridor.label`
- **`_segment_label(corridor, seg_idx)`** → `seg.label`

And **`_exact_facing(grid, from_hex, to_hex)`** — it's defined but never called in `SimAdapter`. If you need it elsewhere, it's a natural `@patch` on `HexGrid`. Otherwise just delete it.

The ones that should stay:

- **`_EVENT_TO_ACTION`** / **`_ACTION_RANK`** — module-level dicts, not object methods
- **`empty_schedule()`** — factory function
- **`schedule_timetable()`** — standalone renderer (takes a DataFrame, not an object)

SCHEDULE_COLUMNS

# 10. QueenCoverage

The main planning object that owns corridors, targets, and assignments. Provides the high-level API: `build_corridor`, `add_target`, `propose_crossing`, `plan`.

In [ ]:
@dataclass
class CampPlan:
    """Budget allocation for pawn roles along a corridor."""
    origin:    list[tuple[int, int]] = field(default_factory=list)  # (hex, tier)
    dest:      list[tuple[int, int]] = field(default_factory=list)
    station:   list[int]             = field(default_factory=list)  # pawn hexes
    target:    Target = None
    n_origin:  int = 0
    n_dest:    int = 0
    n_station: int = 0

    @property
    def total_pawns(self): return self.n_origin + self.n_dest + self.n_station

    def summary(self):
        return (f"{self.n_origin} origin + {self.n_station} station "
                f"+ {self.n_dest} advance = {self.total_pawns} pawns")


In [ ]:
@dataclass
class RosterResult:
    """Output from a shadow-building step."""
    shadows: list       = field(default_factory=list)
    entries: list[dict] = field(default_factory=list)
    timing:  int        = 0  # turns until this group is ready


In [ ]:
@dataclass
class QueenCoverage:
    "Logistics planning layer: corridors, targets, and queen assignments."
    board:      GameBoard
    grid:       object        # HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    profile:    dict          # FOOD_PROFILE_V2

    corridors:   list[Corridor]   = field(default_factory=list)
    targets:     list[Target]     = field(default_factory=list)
    queens:      list[Piece]      = field(default_factory=list)
    assignments: list[Assignment] = field(default_factory=list)

    # ── Lookups ──────────────────────────────────────────────────

    def _countries(self):
        return getattr(self.board, 'terrain', None) and \
               self.board.terrain.fields.get('country')

    def find_corridor(self, start: int, end: int) -> Corridor | None:
        return next((c for c in self.corridors
                     if c.start == start and c.end == end), None)

    def food_level(self, hex: int) -> float:
        tier = int(self.food_tiers[hex]) if 0 <= hex < len(self.food_tiers) else 0
        p = self.profile
        return max(0.0, p.get('pawn_harvest', 1.3) * tier - p.get('pawn_diet', 0.7))

    # ── Targets ──────────────────────────────────────────────────

    def add_target(self, hex: int, purpose: str,
                   priority: float = 1.0, deadline: int | None = None,
                   payoff: float = 0.0, style: StyleCSS = None,
                   glyph: object = None) -> Target:
        t = Target(hex=hex, purpose=purpose, priority=priority,
                   deadline=deadline, payoff=payoff,
                   style=style, glyph=glyph)
        self.targets.append(t)
        return t

    # ── Shadow factories ─────────────────────────────────────────

    def _make_pawn_shadow(self, hex, facing, name,
                          instructions, patrol=True,
                          owner_id=None, flag=None) -> PieceShadow:
        p = self.profile
        ps = PieceShadow(
            id=name, name=name,
            piece_type=PieceType.PAWN, birth_year=0,
            location=hex, facing=facing,
            food=p['pawn_capacity'], food_capacity=p['pawn_capacity'],
            diet=p['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=p['pawn_harvest'],
            sight=p['pawn_sight'],
            owner_id=owner_id, flag=flag)
        ps.instructions = InstructionList(instructions, cursor=0, patrol=patrol)
        return ps

    def _make_queen_shadow(self, queen: Piece, route: list[int],
                           wait_turns: int) -> PieceShadow:
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = self.profile['queen_capacity']
        wait = [Instruction.PAUSE.value] * wait_turns
        march, _ = route_to_instructions(route, self.grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        return qs

    # ── Camp planning ────────────────────────────────────────────

    def _find_camp_hexes(self, center: int, n: int = 2,
                         max_ring: int = 3,
                         exclude: set = None) -> list[tuple[int, int]]:
        """Find best food hexes near a center. Returns (hex, tier) pairs."""
        exclude = exclude or set()
        grid, elevs, tiers = self.grid, self.elevations, self.food_tiers
        cands = []
        for ring in range(1, max_ring + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, center)
                if idx < 0 or idx in grid.invalidRegion: continue
                if idx >= len(elevs) or elevs[idx] <= 0: continue
                if idx in exclude: continue
                tier = int(tiers[idx]) if idx < len(tiers) else 0
                if tier > 0:
                    cands.append((idx, tier))
        cands.sort(key=lambda x: -x[1])
        return cands[:n]

    def _plan_camps(self, corridor: Corridor, budget: int) -> CampPlan:
        """Allocate pawn budget across origin, station, and dest roles."""
        route_set = set(corridor.hexes)
        target = corridor.target

        origin = self._find_camp_hexes(corridor.start, exclude=route_set)
        dest   = self._find_camp_hexes(corridor.end, exclude=route_set)

        station_hexes = [h for seg in corridor.segments
                         if seg.type == 'station' for h in seg.hexes]

        # Purpose-aware weighting
        if target and target.purpose == 'attack':
            max_dest = min(len(dest), 3)
            max_origin = 1
        elif target and target.purpose == 'harvest':
            max_dest = min(len(dest), 1)
            max_origin = 1
        else:
            max_dest = min(len(dest), 2)
            max_origin = min(len(origin), 2)

        n_origin  = min(max_origin, budget)
        n_dest    = min(max_dest, budget - n_origin)
        n_station = min(len(station_hexes), budget - n_origin - n_dest)

        return CampPlan(
            origin=origin[:n_origin], dest=dest[:n_dest],
            station=station_hexes[:n_station], target=target,
            n_origin=n_origin, n_dest=n_dest, n_station=n_station)

    # ── Roster builders (one per role) ───────────────────────────

    def _build_origin_shadows(self, camp: CampPlan,
                              queen: Piece, route: list[int]) -> RosterResult:
        """Origin camp pawns: park near start, HARVEST+GIVE in place."""
        result = RosterResult()
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]

        for i, (hx, tier) in enumerate(camp.origin):
            facing = _facing_toward(self.grid, hx, route[0])
            ps = self._make_pawn_shadow(
                hx, facing, f'camp_o_{i}', harvest_give,
                patrol=True, owner_id=queen.owner_id, flag=queen.flag)
            result.shadows.append(ps)
            result.entries.append(dict(
                piece_type=PieceType.PAWN, hex=hx,
                facing=facing, role='origin_camp'))
        return result

    def _build_advance_shadows(self, camp: CampPlan,
                               queen: Piece,
                               route: list[int]) -> RosterResult:
        """Advance party: sprint to dest camp hexes, then park + harvest."""
        result = RosterResult()
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]

        for i, (hx, tier) in enumerate(camp.dest):
            sprint_path = simple_hex_path(route[0], hx, self.grid, self.elevations)
            if not sprint_path: continue

            sprint_rules, end_facing = route_to_instructions(
                sprint_path, self.grid, 0)

            want_f = _facing_toward(self.grid, hx, route[-1])
            diff = (want_f - end_facing) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))

            park = harvest_give * 80
            instructions = sprint_rules + rot + park

            ps = self._make_pawn_shadow(
                route[0], 0, f'camp_d_{i}', instructions,
                patrol=False, owner_id=queen.owner_id, flag=queen.flag)
            result.shadows.append(ps)
            result.entries.append(dict(
                piece_type=PieceType.PAWN, hex=hx,
                facing=want_f, role='advance_party'))

            arr = math.ceil(len(sprint_path) / 2) + len(rot) + 2
            result.timing = max(result.timing, arr)

        return result

    def _build_station_shadows(self, camp: CampPlan,
                               corridor: Corridor,
                               queen: Piece) -> RosterResult:
        """Mid-route station pawns: park at station hexes, HARVEST+GIVE."""
        result = RosterResult()
        route = corridor.hexes
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]
        pawn_n = 0

        for seg in corridor.segments:
            if seg.type != 'station': continue
            mid_w = (seg.route_start + seg.route_end) // 2
            for hx in seg.hexes:
                if pawn_n >= camp.n_station: break
                facing = _facing_toward(self.grid, hx, route[mid_w])
                ps = self._make_pawn_shadow(
                    hx, facing, f'station_{pawn_n}', harvest_give,
                    patrol=True, owner_id=queen.owner_id, flag=queen.flag)
                result.shadows.append(ps)
                result.entries.append(dict(
                    piece_type=PieceType.PAWN, hex=hx,
                    facing=facing, role='station'))
                pawn_n += 1
        return result

    def _build_bishop_shadows(self, corridor: Corridor,
                              queen: Piece,
                              n_bishops: int) -> RosterResult:
        """Bishop placement: lighthouse MIP first, escort fallback."""
        result = RosterResult()
        route = corridor.hexes

        all_gaps = []
        for seg in corridor.segments:
            if seg.type == 'caravan':
                all_gaps.extend(range(seg.route_start, seg.route_end + 1))

        if not all_gaps or n_bishops <= 0:
            return result

        # Try lighthouse placement
        lh = optimal_lighthouse_placement(
            all_gaps, route, self.grid, self.elevations,
            n_bishops=n_bishops, profile=self.profile)

        if lh and lh['feasible']:
            for pl in lh['placements']:
                bs, arr = self._make_bishop_shadow(
                    pl['hex'], pl['serves_route_w'], route,
                    self.profile, queen,
                    f'blh_{pl["bishop"]}', f'Bishop_{pl["bishop"]}')
                if bs:
                    result.shadows.append(bs)
                    result.entries.append(dict(
                        piece_type=PieceType.BISHOP, hex=pl['hex'],
                        facing=bs.facing, role='lighthouse'))
                    result.timing = max(result.timing, arr)
        else:
            # Escort fallback
            gap_segs = _split_contiguous(all_gaps)
            for bi, gseg in enumerate(gap_segs[:n_bishops]):
                mid_w = gseg[len(gseg) // 2]
                park_hex = route[mid_w]
                bs, arr = self._make_bishop_shadow(
                    park_hex, gseg, route, self.profile, queen,
                    f'besc_{bi}', f'Escort_{bi}')
                if bs:
                    result.shadows.append(bs)
                    result.entries.append(dict(
                        piece_type=PieceType.BISHOP, hex=park_hex,
                        facing=bs.facing, role='escort'))
                    result.timing = max(result.timing, arr)

        return result

    # ── Corridor metrics ─────────────────────────────────────────

    def _metrics_from_segments(self, segments: list[Segment]) -> dict:
        b_cap = self.profile.get('bishop_capacity', 140.0)
        if not segments:
            return dict(n_stations=0, n_gaps=0, fuel_needed=0.0,
                        bishop_min_fuel=0.0, duration=0, valid=True)

        n_stations = sum(1 for s in segments if s.type == 'station')
        gap_segs   = [s for s in segments if s.type == 'caravan']
        n_gaps     = sum(s.gap_len for s in gap_segs)
        fuel_needed = sum(s.fuel_needed for s in gap_segs)
        duration = int(b_cap / max(fuel_needed, 0.1)) if fuel_needed > 0 else 0

        return dict(n_stations=n_stations, n_gaps=n_gaps,
                    fuel_needed=fuel_needed, bishop_min_fuel=b_cap,
                    duration=duration, valid=True)

    # ── Corridor construction ────────────────────────────────────

    def build_corridor(self, start: int, end: int,
                       kind: CorridorType = CorridorType.TEMPORARY,
                       n_pawns: int = 4, n_bishops: int = 1,
                       target: Target = None,
                       style: StyleCSS = None,
                       glyph: DiagramGlyphs = None) -> Corridor | None:
        existing = self.find_corridor(start, end)
        if existing:
            print(f"build_corridor: reusing ⬡{start}→⬡{end}")
            return existing

        p = self.profile
        pcr = pawn_covered_route(
            start, end, grid=self.grid, elevations=self.elevations,
            food_tiers=self.food_tiers,
            pawn_harvest=p.get('pawn_harvest', 1.3),
            pawn_diet=p.get('pawn_diet', 0.7),
            queen_diet=p.get('queen_diet', 7.0),
            cone_range=p.get('pawn_sight', 4),
            n_bishops=1,
            bishop_capacity=p.get('bishop_capacity', 140.0),
            bishop_diet=p.get('bishop_diet', 2.5),
            countries=self._countries())
        if pcr is None:
            print(f"build_corridor: no route ⬡{start}→⬡{end}")
            return None

        segments = find_stations_v3(pcr, n_pawns=n_pawns, n_bishops=1, profile=p)
        metrics = self._metrics_from_segments(segments)

        if n_bishops > 1 and metrics['fuel_needed'] > 0:
            metrics['bishop_min_fuel'] = p.get('bishop_capacity', 140.0) * n_bishops
            metrics['duration'] = int(metrics['bishop_min_fuel'] /
                                      max(metrics['fuel_needed'], 0.1))

        corridor = Corridor(hexes=pcr['route'], kind=kind,
                            segments=segments, target=target,
                            style=style, glyph=glyph, **metrics)
        self.corridors.append(corridor)

        tag = '✅' if corridor.feasible else '❌'
        print(f"Corridor ⬡{start}→⬡{end} ({kind.value}): "
              f"{corridor.length}hex, {corridor.n_stations} stations, "
              f"{corridor.n_gaps} gaps, fuel={corridor.fuel_needed:.1f} {tag}")
        return corridor

    # ── Bishop / congestion helpers ──────────────────────────────

    def assign_bishop(self, corridor: Corridor, bishop: Piece):
        fuel = getattr(bishop, 'food', self.profile.get('bishop_capacity', 140.0))
        corridor.bishop_min_fuel = min(corridor.bishop_min_fuel, fuel)

    def update_congestion(self):
        for c in self.corridors:
            rs = set(c.hexes)
            c.congestion = sum(
                1 for kingdom in self.board.kingdoms
                for settlement in kingdom.settlements
                for piece in settlement.citizens
                if piece.location in rs)

    # ── Crossing proposal ────────────────────────────────────────

    def propose_crossing(self, corridor: Corridor, queen: Piece,
                         available_pawns: int = 5,
                         available_bishops: int = 2,
                         max_retries: int = 3,
                         n_turns: int = 60) -> CrossingPlan:
        """Propose a validated crossing: assemble roster, simulate, retry."""
        camp = self._plan_camps(corridor, available_pawns)
        print(f"  Camp plan: {camp.summary()}")
        print(f"  Origin: {camp.origin}  Dest: {camp.dest}")

        best_plan = None
        route = corridor.hexes

        for attempt in range(max_retries + 1):
            extra_wait = attempt * 2

            # ── Assemble roster ──
            origin_r  = self._build_origin_shadows(camp, queen, route)
            advance_r = self._build_advance_shadows(camp, queen, route)
            station_r = self._build_station_shadows(camp, corridor, queen)
            bishop_r  = self._build_bishop_shadows(corridor, queen, available_bishops)

            queen_start = max(bishop_r.timing, advance_r.timing) + extra_wait
            qs = self._make_queen_shadow(queen, route, queen_start)

            shadows = ([qs] + origin_r.shadows + advance_r.shadows
                       + station_r.shadows + bishop_r.shadows)
            roster = ([dict(piece_type=PieceType.QUEEN, hex=route[0],
                            facing=qs.facing, role='queen')]
                      + origin_r.entries + advance_r.entries
                      + station_r.entries + bishop_r.entries)

            # ── Stamp ownership ──
            for s in shadows:
                s.owner_id = queen.owner_id
                s.flag = queen.flag

            # ── Simulate ──
            plan = CrossingPlan(
                corridor=corridor, roster=roster,
                queen_start_turn=queen_start)
            plan.simulate(
                shadows=shadows, queen_shadow=qs,
                grid=self.grid, elevations=self.elevations,
                food_tiers=self.food_tiers, profile=self.profile,
                countries=self._countries(), n_turns=n_turns)

            best_plan = plan

            if plan.feasible:
                roles = Counter(r['role'] for r in roster)
                print(f"✅ Crossing (attempt {attempt}): roles={dict(roles)}, "
                      f"start=t{queen_start}, arrive=t{plan.queen_arrival}, "
                      f"min_food={plan.food_margin:.1f}")
                return plan

            print(f"❌ Attempt {attempt}: arrive={'yes' if plan.queen_arrival < n_turns else 'no'}, "
                  f"min_food={plan.food_margin:.1f}, wait={queen_start}t")

        print(f"⚠️ All {max_retries + 1} attempts failed")
        return best_plan


In [ ]:
def _split_contiguous(indices):
    """Split a sorted list of ints into contiguous runs."""
    if not indices: return []
    segs, cur = [], [indices[0]]
    for g in indices[1:]:
        if g == cur[-1] + 1: cur.append(g)
        else: segs.append(cur); cur = [g]
    segs.append(cur)
    return segs


@patch
def _make_bishop_shadow(self: QueenCoverage, park_hex, served_ws, route,
                        profile, queen, shadow_id, shadow_name):
    """Create a bishop shadow that sprints to park_hex, faces the gap, parks.

    Returns (PieceShadow, arrival_turn) or (None, 0) if no path.
    """
    bp = simple_hex_path(route[0], park_hex, self.grid, self.elevations)
    if not bp: return None, 0

    sprint, end_f = route_to_instructions(bp, self.grid, 0)

    # Face toward the middle of the served gap for max sight coverage
    mid_w = served_ws[len(served_ws) // 2]
    want_f = _facing_toward(self.grid, park_hex, route[mid_w])
    d = (want_f - end_f) % 6
    rot = ([Instruction.ROT_R.value] * d if d <= 3
           else [Instruction.ROT_L.value] * (6 - d))

    park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 60

    bs = PieceShadow(
        id=shadow_id, name=shadow_name,
        piece_type=PieceType.BISHOP, birth_year=0,
        location=route[0], facing=0,
        food=profile['bishop_capacity'],
        food_capacity=profile['bishop_capacity'],
        diet=profile.get('bishop_diet', 3.5),
        health=100, max_health=100, move_strength=3,
        harvest_strength=profile.get('bishop_harvest', 0.7),
        sight=profile['pawn_sight'],
        owner_id=queen.owner_id, flag=queen.flag)
    bs.instructions = InstructionList(sprint + rot + park, cursor=0, patrol=False)

    arrival = math.ceil(len(bp) / 3) + 2  # bishop speed 3, +safety
    return bs, arrival

## 11. Crossing Proposal

The validation layer. Creates shadow pieces (PieceShadow clones) and runs a throwaway FoodSimulator. Pawn roles: origin camp (park + feed queen while she waits), advance party (sprint to dest, park, harvest before queen arrives), mid-route stations (from corridor segments). Queen departs only after bishops are parked AND advance party has arrived.

def propose_crossing(self: QueenCoverage, corridor: Corridor,
                     queen: Piece,
                     available_pawns: int = 5,
                     available_bishops: int = 2,
                     max_retries: int = 3,
                     n_turns: int = 60) -> CrossingPlan:
    """Propose a validated crossing plan with origin + destination camps.

    Pawn roles (allocated from available_pawns budget):
      origin_camp:    park near route start, feed queen while she waits
      advance_party:  sprint to dest camp hexes BEFORE queen departs
      station:        mid-route food stations from corridor segments

    Queen departs only after bishops are parked AND advance party has
    arrived + harvested at least once. Origin camp keeps her full until then.
    """
    route = corridor.hexes
    segments = corridor.segments
    profile = self.profile
    grid = self.grid
    elevs = self.elevations
    food_tiers = self.food_tiers

    # ── Find best food hexes near an endpoint ──────────────────────────
    def _find_camp_hexes(center, n=2, max_ring=3):
        route_set = set(route)
        cands = []
        for ring in range(1, max_ring + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, center)
                if idx < 0 or idx in grid.invalidRegion: continue
                if idx >= len(elevs) or elevs[idx] <= 0: continue
                if idx in route_set: continue
                tier = int(food_tiers[idx]) if idx < len(food_tiers) else 0
                if tier > 0:
                    cands.append((idx, tier))
        cands.sort(key=lambda x: -x[1])
        return cands[:n]

    origin_camps = _find_camp_hexes(route[0])
    dest_camps   = _find_camp_hexes(route[-1])

    # ── Pawn budget allocation ─────────────────────────────────────────
    # Count mid-station slots from corridor segments
    station_hexes = []
    for seg in segments:
        if seg['type'] == 'station':
            station_hexes.extend(seg['hexes'])

    n_origin  = min(len(origin_camps), 2)
    n_dest    = min(len(dest_camps), 2)
    n_station = min(len(station_hexes), available_pawns - n_origin - n_dest)
    # Clamp to budget
    total_pawns = n_origin + n_dest + n_station
    if total_pawns > available_pawns:
        # Trim stations first, then dest, then origin
        n_station = max(0, available_pawns - n_origin - n_dest)
        if n_origin + n_dest > available_pawns:
            n_dest = max(0, available_pawns - n_origin)

    print(f"  Camp plan: {n_origin} origin + {n_station} station "
          f"+ {n_dest} advance → {n_origin + n_station + n_dest}/{available_pawns} pawns")
    print(f"  Origin camps: {[(h, f'T{t}') for h, t in origin_camps[:n_origin]]}")
    print(f"  Dest camps:   {[(h, f'T{t}') for h, t in dest_camps[:n_dest]]}")

    best_plan = None

    for attempt in range(max_retries + 1):
        extra_wait = attempt * 2
        roster, shadows = [], []
        pawn_n = 0

        # ── 1. Queen shadow (instructions set in step 6) ──────────────
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = profile['queen_capacity']
        roster.append(dict(piece_type=PieceType.QUEEN, hex=route[0],
                           facing=qs.facing, role='queen'))

        # ── 2. Origin camp pawns (park + HARVEST+GIVE) ────────────────
        for hx, tier in origin_camps[:n_origin]:
            facing = _facing_toward(grid, hx, route[0])
            ps = PieceShadow(
                id=f'camp_o_{pawn_n}', name=f'OriginPawn_{pawn_n}',
                piece_type=PieceType.PAWN, birth_year=0,
                location=hx, facing=facing,
                food=profile['pawn_capacity'],
                food_capacity=profile['pawn_capacity'],
                diet=profile['pawn_diet'], health=100, max_health=100,
                move_strength=2, harvest_strength=profile['pawn_harvest'],
                sight=profile['pawn_sight'],
                owner_id=queen.owner_id, flag=queen.flag)
            ps.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
            shadows.append(ps)
            roster.append(dict(piece_type=PieceType.PAWN, hex=hx,
                               facing=facing, role='origin_camp'))
            pawn_n += 1

        # ── 3. Advance party (sprint to dest camp, then park) ─────────
        advance_arrival = 0
        for hx, tier in dest_camps[:n_dest]:
            sprint_path = simple_hex_path(route[0], hx, grid, elevs)
            if not sprint_path:
                continue
            sprint_rules, end_facing = route_to_instructions(
                sprint_path, grid, 0)

            want_f = _facing_toward(grid, hx, route[-1])
            diff = (want_f - end_facing) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))

            park = [Instruction.HARVEST.value, Instruction.GIVE.value] * 80

            ps = PieceShadow(
                id=f'camp_d_{pawn_n}', name=f'DestPawn_{pawn_n}',
                piece_type=PieceType.PAWN, birth_year=0,
                location=route[0], facing=0,
                food=profile['pawn_capacity'],
                food_capacity=profile['pawn_capacity'],
                diet=profile['pawn_diet'], health=100, max_health=100,
                move_strength=2, harvest_strength=profile['pawn_harvest'],
                sight=profile['pawn_sight'],
                owner_id=queen.owner_id, flag=queen.flag)
            ps.instructions = InstructionList(
                sprint_rules + rot + park, cursor=0, patrol=False)
            shadows.append(ps)
            roster.append(dict(piece_type=PieceType.PAWN, hex=hx,
                               facing=want_f, role='advance_party'))
            pawn_n += 1

            # Estimate arrival: path_len / speed + rotate + 1 harvest
            arr = math.ceil(len(sprint_path) / 2) + len(rot) + 2
            advance_arrival = max(advance_arrival, arr)

        # ── 4. Mid-route station pawns ────────────────────────────────
        for seg in segments:
            if seg['type'] != 'station': continue
            for hx in seg['hexes']:
                if pawn_n >= n_origin + n_dest + n_station: break
                mid_w = (seg['min_w'] + seg['max_w']) // 2
                facing = _facing_toward(grid, hx, route[mid_w])
                ps = PieceShadow(
                    id=f'station_{pawn_n}', name=f'StationPawn_{pawn_n}',
                    piece_type=PieceType.PAWN, birth_year=0,
                    location=hx, facing=facing,
                    food=profile['pawn_capacity'],
                    food_capacity=profile['pawn_capacity'],
                    diet=profile['pawn_diet'], health=100, max_health=100,
                    move_strength=2, harvest_strength=profile['pawn_harvest'],
                    sight=profile['pawn_sight'],
                    owner_id=queen.owner_id, flag=queen.flag)
                ps.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
                shadows.append(ps)
                roster.append(dict(piece_type=PieceType.PAWN, hex=hx,
                                   facing=facing, role='station'))
                pawn_n += 1

        # ── 5. Bishops (lighthouse MIP or escort fallback) ────────────
        all_gaps = []
        for seg in segments:
            if seg['type'] == 'caravan':
                all_gaps.extend(range(seg['gap_start'], seg['gap_end'] + 1))

        bishop_ready = 0
        bishop_mode = 'none'

        if all_gaps and available_bishops > 0:
            lh = optimal_lighthouse_placement(
                all_gaps, route, grid, elevs,
                n_bishops=available_bishops, profile=profile)

            if lh and lh['feasible']:
                bishop_mode = 'lighthouse'
                for pl in lh['placements']:
                    bs, arr = self._make_bishop_shadow(
                        pl['hex'], pl['serves_route_w'], route,
                        profile, queen, f'blh_{pl["bishop"]}',
                        f'Bishop_{pl["bishop"]}')
                    if bs:
                        shadows.append(bs)
                        roster.append(dict(piece_type=PieceType.BISHOP,
                                           hex=pl['hex'], facing=bs.facing,
                                           role='lighthouse'))
                        bishop_ready = max(bishop_ready, arr)
            else:
                bishop_mode = 'escort'
                gap_segs = _split_contiguous(all_gaps)
                for bi, gseg in enumerate(gap_segs[:available_bishops]):
                    mid_w = gseg[len(gseg) // 2]
                    park_hex = route[mid_w]
                    bs, arr = self._make_bishop_shadow(
                        park_hex, gseg, route, profile, queen,
                        f'besc_{bi}', f'Escort_{bi}')
                    if bs:
                        shadows.append(bs)
                        roster.append(dict(piece_type=PieceType.BISHOP,
                                           hex=park_hex, facing=bs.facing,
                                           role='escort'))
                        bishop_ready = max(bishop_ready, arr)

        # ── 6. Queen timing: depart after bishops + advance party ─────
        queen_start = max(bishop_ready, advance_arrival) + extra_wait
        wait  = [Instruction.PAUSE.value] * queen_start
        march, _ = route_to_instructions(route, grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        shadows.insert(0, qs)

        # ── 7. Ownership + squad ──────────────────────────────────────
        for s in shadows:
            s.owner_id = queen.owner_id
            s.flag = queen.flag

        shadow_squad = Squad.squads(1, queen.flag)[0]
        shadow_squad.pieces = list(shadows)

        initial = {s.id: dict(location=s.location, facing=s.facing,
                               food=s.food, health=s.health)
                   for s in shadows}

        # ── 8. Simulate ──────────────────────────────────────────────
        sim = FoodSimulator(
            grid=grid, elevations=elevs,
            food_tiers=food_tiers, pieces=shadows,
            countries=self._countries(),
            squads=[shadow_squad],
            callbacks={shadow_squad.id: food_redistribute_adjacent},
        )
        sim.run(num_turns=n_turns)

        # ── 9. Evaluate ──────────────────────────────────────────────
        q_snap = [s['piece_states'].get(qs.id) for s in sim.snapshots]
        arrived = any(s and s['location'] == route[-1] for s in q_snap)
        alive   = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
        min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                       default=0)

        arrival_turn = n_turns
        for i, s in enumerate(q_snap):
            if s and s['location'] == route[-1]:
                arrival_turn = i + 1; break

        bishop_fuel = 0
        if sim.snapshots:
            for s in shadows:
                if s.piece_type == PieceType.BISHOP:
                    ls = sim.snapshots[-1]['piece_states'].get(s.id)
                    if ls:
                        bishop_fuel += profile['bishop_capacity'] - ls.get('food', 0)

        adapter = SimAdapter(sim, corridors=[corridor])
        schedule = adapter.extract(schedule_id=f'crossing_v{attempt}',
                                    initial_states=initial)

        feasible = arrived and alive
        plan = CrossingPlan(
            corridor=corridor, roster=roster,
            queen_start_turn=queen_start, schedule=schedule,
            feasible=feasible, queen_arrival=arrival_turn,
            food_margin=min_food, bishop_fuel_used=bishop_fuel)
        best_plan = plan

        if feasible:
            roles = {}
            for r in roster:
                roles[r['role']] = roles.get(r['role'], 0) + 1
            print(f"✅ Crossing (attempt {attempt}, {bishop_mode}): "
                  f"roles={roles}, start=t{queen_start}, "
                  f"arrive=t{arrival_turn}, min_food={min_food:.1f}")
            return plan

        reason = 'died' if not alive else 'never arrived'
        last_hex = None
        for st in reversed(q_snap):
            if st and st['health'] > 0: last_hex = st['location']; break
        print(f"❌ Attempt {attempt} ({bishop_mode}): {reason}, "
              f"last=⬡{last_hex}, min_food={min_food:.1f}, "
              f"wait={queen_start}t")

    print(f"⚠️ All {max_retries + 1} attempts failed")
    return best_plan

propose_crossing is rather larger function and we have been refactoring a bit. should some of this be put into the CoveragePlan class? How can we make this more maintainable

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, `propose_crossing` is doing about 5 distinct jobs in one function. Here's how I'd break it up:

**1. Shadow factory** — The repetitive `PieceShadow(...)` creation is begging to be a helper:

```python
def _make_pawn_shadow(self, hex, facing, name, role_instructions,
                      patrol=True) -> PieceShadow:
    p = self.profile
    ps = PieceShadow(
        id=f'{name}', name=name,
        piece_type=PieceType.PAWN, birth_year=0,
        location=hex, facing=facing,
        food=p['pawn_capacity'], food_capacity=p['pawn_capacity'],
        diet=p['pawn_diet'], health=100, max_health=100,
        move_strength=2, harvest_strength=p['pawn_harvest'],
        sight=p['pawn_sight'],
        owner_id=None, flag=None)  # stamped later
    ps.instructions = InstructionList(role_instructions, cursor=0, patrol=patrol)
    return ps
```

**2. Camp planning** — finding camp hexes + budget allocation:

```python
@dataclass
class CampPlan:
    origin:  list[tuple[int, int]]   # (hex, tier) pairs
    dest:    list[tuple[int, int]]
    station: list[int]               # station pawn hexes
    n_origin:  int = 0
    n_dest:    int = 0
    n_station: int = 0

    @property
    def total_pawns(self): return self.n_origin + self.n_dest + self.n_station
```

Then a method:

```python
def _plan_camps(self, corridor: Corridor, budget: int) -> CampPlan:
    origin = self._find_camp_hexes(corridor.hexes[0])
    dest   = self._find_camp_hexes(corridor.hexes[-1])
    station_hexes = [h for seg in corridor.segments
                     if seg.type == 'station' for h in seg.hexes]

    n_origin  = min(len(origin), 2)
    n_dest    = min(len(dest), 2)
    n_station = min(len(station_hexes), budget - n_origin - n_dest)
    # Clamp overflow
    if n_origin + n_dest > budget:
        n_dest = max(0, budget - n_origin)
    n_station = max(0, n_station)

    return CampPlan(origin=origin[:n_origin], dest=dest[:n_dest],
                    station=station_hexes[:n_station],
                    n_origin=n_origin, n_dest=n_dest, n_station=n_station)
```

**3. Roster assembly** — one method per role, returns `(shadows, roster_entries, timing)`:

```python
def _build_origin_shadows(self, camp: CampPlan, queen, route):
    ...  # returns (shadows, roster_entries)

def _build_advance_shadows(self, camp: CampPlan, queen, route):
    ...  # returns (shadows, roster_entries, advance_arrival)

def _build_station_shadows(self, camp: CampPlan, corridor, queen):
    ...  # returns (shadows, roster_entries)

def _build_bishop_shadows(self, corridor, queen, n_bishops):
    ...  # returns (shadows, roster_entries, bishop_ready, mode)
```

**4. `CrossingPlan.simulate()`** — move sim + eval onto the plan itself:

```python
@patch
def simulate(self: CrossingPlan, grid, elevations, food_tiers,
             profile, countries=None, n_turns=60) -> 'CrossingPlan':
    """Run FoodSimulator on the roster, populate feasible/arrival/margins."""
    # Build shadows from roster...
    # Run sim...
    # Extract queen snapshots, check arrived + alive...
    self.feasible = arrived and alive
    self.queen_arrival = arrival_turn
    self.food_margin = min_food
    self.bishop_fuel_used = bishop_fuel
    return self
```

**5. Thin orchestrator** — `propose_crossing` becomes glue:

```python
def propose_crossing(self, corridor, queen,
                     available_pawns=5, available_bishops=2,
                     max_retries=3, n_turns=60) -> CrossingPlan:
    camp = self._plan_camps(corridor, available_pawns)

    for attempt in range(max_retries + 1):
        extra_wait = attempt * 2
        shadows, roster = [], []

        # Assemble roster
        for builder in (self._build_origin_shadows,
                        self._build_station_shadows):
            s, r = builder(camp, queen, corridor)
            shadows += s; roster += r

        adv_s, adv_r, adv_arrival = self._build_advance_shadows(
            camp, queen, corridor.hexes)
        shadows += adv_s; roster += adv_r

        b_s, b_r, b_ready, b_mode = self._build_bishop_shadows(
            corridor, queen, available_bishops)
        shadows += b_s; roster += b_r

        # Queen timing
        queen_start = max(b_ready, adv_arrival) + extra_wait
        qs = self._make_queen_shadow(queen, corridor.hexes, queen_start)
        shadows.insert(0, qs)
        roster.insert(0, dict(piece_type=PieceType.QUEEN,
                              hex=corridor.hexes[0], role='queen'))

        # Simulate
        plan = CrossingPlan(corridor=corridor, roster=roster,
                            queen_start_turn=queen_start)
        plan.simulate(self.grid, self.elevations, self.food_tiers,
                      self.profile, n_turns=n_turns)

        if plan.feasible:
            return plan

    return plan  # best failed attempt
```

This gives you:

| Piece | Responsibility |
|---|---|
| `_make_pawn_shadow` | DRY shadow creation |
| `CampPlan` | Testable budget allocation |
| `_build_*_shadows` | One role = one method |
| `CrossingPlan.simulate` | Plan owns its own validation |
| `propose_crossing` | 30-line orchestrator |

Also need to fix the old dict-style segment access (`seg['type']` → `seg.type`, `seg['hexes']` → `seg.hexes`, etc.) throughout.

Want to tackle these one at a time, or should I write the whole refactored version?

Should CampPlan have target built in?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good instinct. The dest camp is placed near the target hex, and the target's `purpose` could naturally influence budget allocation — an `'attack'` target might want more pawns at dest for a staging area, while `'harvest'` wants them spread along the route.

But I'd keep it as a **reference, not ownership** — Target already lives on Corridor and Assignment. CampPlan just needs to see it:

```python
@dataclass
class CampPlan:
    origin:    list[tuple[int, int]]   # (hex, tier) pairs
    dest:      list[tuple[int, int]]
    station:   list[int]               # station pawn hexes
    target:    Target = None           # reference, not owned
    n_origin:  int = 0
    n_dest:    int = 0
    n_station: int = 0

    @property
    def total_pawns(self): return self.n_origin + self.n_dest + self.n_station
```

Then `_plan_camps` can use `target.purpose` to bias the split:

```python
def _plan_camps(self, corridor, budget) -> CampPlan:
    target = corridor.target
    # Attack: heavier dest presence. Harvest: heavier station presence.
    dest_weight = 2 if target and target.purpose == 'attack' else 1
    ...
```

This way CampPlan is testable on its own, but purpose-aware when a target is provided.

Can you refactor the whole thing?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the full refactor — broken into clean, testable pieces:

```python
@dataclass
class CampPlan:
    """Budget allocation for pawn roles along a corridor."""
    origin:    list[tuple[int, int]] = field(default_factory=list)  # (hex, tier)
    dest:      list[tuple[int, int]] = field(default_factory=list)
    station:   list[int]             = field(default_factory=list)  # pawn hexes
    target:    Target = None
    n_origin:  int = 0
    n_dest:    int = 0
    n_station: int = 0

    @property
    def total_pawns(self): return self.n_origin + self.n_dest + self.n_station

    def summary(self):
        return (f"{self.n_origin} origin + {self.n_station} station "
                f"+ {self.n_dest} advance = {self.total_pawns} pawns")
```

```python
@dataclass
class RosterResult:
    """Output from a shadow-building step."""
    shadows: list       = field(default_factory=list)
    entries: list[dict] = field(default_factory=list)
    timing:  int        = 0  # turns until this group is ready
```

```python
@dataclass
class QueenCoverage:
    "Logistics planning layer: corridors, targets, and queen assignments."
    board:      GameBoard
    grid:       object        # HexGrid
    elevations: np.ndarray
    food_tiers: np.ndarray
    profile:    dict          # FOOD_PROFILE_V2

    corridors:   list[Corridor]   = field(default_factory=list)
    targets:     list[Target]     = field(default_factory=list)
    queens:      list[Piece]      = field(default_factory=list)
    assignments: list[Assignment] = field(default_factory=list)

    # ── Lookups ──────────────────────────────────────────────────

    def _countries(self):
        return getattr(self.board, 'terrain', None) and \
               self.board.terrain.fields.get('country')

    def find_corridor(self, start: int, end: int) -> Corridor | None:
        return next((c for c in self.corridors
                     if c.start == start and c.end == end), None)

    def food_level(self, hex: int) -> float:
        tier = int(self.food_tiers[hex]) if 0 <= hex < len(self.food_tiers) else 0
        p = self.profile
        return max(0.0, p.get('pawn_harvest', 1.3) * tier - p.get('pawn_diet', 0.7))

    # ── Targets ──────────────────────────────────────────────────

    def add_target(self, hex: int, purpose: str,
                   priority: float = 1.0, deadline: int | None = None,
                   payoff: float = 0.0, style: StyleCSS = None,
                   glyph: object = None) -> Target:
        t = Target(hex=hex, purpose=purpose, priority=priority,
                   deadline=deadline, payoff=payoff,
                   style=style, glyph=glyph)
        self.targets.append(t)
        return t

    # ── Shadow factories ─────────────────────────────────────────

    def _make_pawn_shadow(self, hex, facing, name,
                          instructions, patrol=True,
                          owner_id=None, flag=None) -> PieceShadow:
        p = self.profile
        ps = PieceShadow(
            id=name, name=name,
            piece_type=PieceType.PAWN, birth_year=0,
            location=hex, facing=facing,
            food=p['pawn_capacity'], food_capacity=p['pawn_capacity'],
            diet=p['pawn_diet'], health=100, max_health=100,
            move_strength=2, harvest_strength=p['pawn_harvest'],
            sight=p['pawn_sight'],
            owner_id=owner_id, flag=flag)
        ps.instructions = InstructionList(instructions, cursor=0, patrol=patrol)
        return ps

    def _make_queen_shadow(self, queen: Piece, route: list[int],
                           wait_turns: int) -> PieceShadow:
        qs = PieceShadow.from_piece(queen)
        qs.location = route[0]
        qs.food = self.profile['queen_capacity']
        wait = [Instruction.PAUSE.value] * wait_turns
        march, _ = route_to_instructions(route, self.grid, qs.facing)
        qs.instructions = InstructionList(wait + march, cursor=0, patrol=False)
        return qs

    # ── Camp planning ────────────────────────────────────────────

    def _find_camp_hexes(self, center: int, n: int = 2,
                         max_ring: int = 3,
                         exclude: set = None) -> list[tuple[int, int]]:
        """Find best food hexes near a center. Returns (hex, tier) pairs."""
        exclude = exclude or set()
        grid, elevs, tiers = self.grid, self.elevations, self.food_tiers
        cands = []
        for ring in range(1, max_ring + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, center)
                if idx < 0 or idx in grid.invalidRegion: continue
                if idx >= len(elevs) or elevs[idx] <= 0: continue
                if idx in exclude: continue
                tier = int(tiers[idx]) if idx < len(tiers) else 0
                if tier > 0:
                    cands.append((idx, tier))
        cands.sort(key=lambda x: -x[1])
        return cands[:n]

    def _plan_camps(self, corridor: Corridor, budget: int) -> CampPlan:
        """Allocate pawn budget across origin, station, and dest roles."""
        route_set = set(corridor.hexes)
        target = corridor.target

        origin = self._find_camp_hexes(corridor.start, exclude=route_set)
        dest   = self._find_camp_hexes(corridor.end, exclude=route_set)

        station_hexes = [h for seg in corridor.segments
                         if seg.type == 'station' for h in seg.hexes]

        # Purpose-aware weighting
        if target and target.purpose == 'attack':
            max_dest = min(len(dest), 3)
            max_origin = 1
        elif target and target.purpose == 'harvest':
            max_dest = min(len(dest), 1)
            max_origin = 1
        else:
            max_dest = min(len(dest), 2)
            max_origin = min(len(origin), 2)

        n_origin  = min(max_origin, budget)
        n_dest    = min(max_dest, budget - n_origin)
        n_station = min(len(station_hexes), budget - n_origin - n_dest)

        return CampPlan(
            origin=origin[:n_origin], dest=dest[:n_dest],
            station=station_hexes[:n_station], target=target,
            n_origin=n_origin, n_dest=n_dest, n_station=n_station)

    # ── Roster builders (one per role) ───────────────────────────

    def _build_origin_shadows(self, camp: CampPlan,
                              queen: Piece, route: list[int]) -> RosterResult:
        """Origin camp pawns: park near start, HARVEST+GIVE in place."""
        result = RosterResult()
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]

        for i, (hx, tier) in enumerate(camp.origin):
            facing = _facing_toward(self.grid, hx, route[0])
            ps = self._make_pawn_shadow(
                hx, facing, f'camp_o_{i}', harvest_give,
                patrol=True, owner_id=queen.owner_id, flag=queen.flag)
            result.shadows.append(ps)
            result.entries.append(dict(
                piece_type=PieceType.PAWN, hex=hx,
                facing=facing, role='origin_camp'))
        return result

    def _build_advance_shadows(self, camp: CampPlan,
                               queen: Piece,
                               route: list[int]) -> RosterResult:
        """Advance party: sprint to dest camp hexes, then park + harvest."""
        result = RosterResult()
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]

        for i, (hx, tier) in enumerate(camp.dest):
            sprint_path = simple_hex_path(route[0], hx, self.grid, self.elevations)
            if not sprint_path: continue

            sprint_rules, end_facing = route_to_instructions(
                sprint_path, self.grid, 0)

            want_f = _facing_toward(self.grid, hx, route[-1])
            diff = (want_f - end_facing) % 6
            rot = ([Instruction.ROT_R.value] * diff if diff <= 3
                   else [Instruction.ROT_L.value] * (6 - diff))

            park = harvest_give * 80
            instructions = sprint_rules + rot + park

            ps = self._make_pawn_shadow(
                route[0], 0, f'camp_d_{i}', instructions,
                patrol=False, owner_id=queen.owner_id, flag=queen.flag)
            result.shadows.append(ps)
            result.entries.append(dict(
                piece_type=PieceType.PAWN, hex=hx,
                facing=want_f, role='advance_party'))

            arr = math.ceil(len(sprint_path) / 2) + len(rot) + 2
            result.timing = max(result.timing, arr)

        return result

    def _build_station_shadows(self, camp: CampPlan,
                               corridor: Corridor,
                               queen: Piece) -> RosterResult:
        """Mid-route station pawns: park at station hexes, HARVEST+GIVE."""
        result = RosterResult()
        route = corridor.hexes
        harvest_give = [Instruction.HARVEST.value, Instruction.GIVE.value]
        pawn_n = 0

        for seg in corridor.segments:
            if seg.type != 'station': continue
            mid_w = (seg.route_start + seg.route_end) // 2
            for hx in seg.hexes:
                if pawn_n >= camp.n_station: break
                facing = _facing_toward(self.grid, hx, route[mid_w])
                ps = self._make_pawn_shadow(
                    hx, facing, f'station_{pawn_n}', harvest_give,
                    patrol=True, owner_id=queen.owner_id, flag=queen.flag)
                result.shadows.append(ps)
                result.entries.append(dict(
                    piece_type=PieceType.PAWN, hex=hx,
                    facing=facing, role='station'))
                pawn_n += 1
        return result

    def _build_bishop_shadows(self, corridor: Corridor,
                              queen: Piece,
                              n_bishops: int) -> RosterResult:
        """Bishop placement: lighthouse MIP first, escort fallback."""
        result = RosterResult()
        route = corridor.hexes

        all_gaps = []
        for seg in corridor.segments:
            if seg.type == 'caravan':
                all_gaps.extend(range(seg.route_start, seg.route_end + 1))

        if not all_gaps or n_bishops <= 0:
            return result

        # Try lighthouse placement
        lh = optimal_lighthouse_placement(
            all_gaps, route, self.grid, self.elevations,
            n_bishops=n_bishops, profile=self.profile)

        if lh and lh['feasible']:
            for pl in lh['placements']:
                bs, arr = self._make_bishop_shadow(
                    pl['hex'], pl['serves_route_w'], route,
                    self.profile, queen,
                    f'blh_{pl["bishop"]}', f'Bishop_{pl["bishop"]}')
                if bs:
                    result.shadows.append(bs)
                    result.entries.append(dict(
                        piece_type=PieceType.BISHOP, hex=pl['hex'],
                        facing=bs.facing, role='lighthouse'))
                    result.timing = max(result.timing, arr)
        else:
            # Escort fallback
            gap_segs = _split_contiguous(all_gaps)
            for bi, gseg in enumerate(gap_segs[:n_bishops]):
                mid_w = gseg[len(gseg) // 2]
                park_hex = route[mid_w]
                bs, arr = self._make_bishop_shadow(
                    park_hex, gseg, route, self.profile, queen,
                    f'besc_{bi}', f'Escort_{bi}')
                if bs:
                    result.shadows.append(bs)
                    result.entries.append(dict(
                        piece_type=PieceType.BISHOP, hex=park_hex,
                        facing=bs.facing, role='escort'))
                    result.timing = max(result.timing, arr)

        return result

    # ── Corridor metrics ─────────────────────────────────────────

    def _metrics_from_segments(self, segments: list[Segment]) -> dict:
        b_cap = self.profile.get('bishop_capacity', 140.0)
        if not segments:
            return dict(n_stations=0, n_gaps=0, fuel_needed=0.0,
                        bishop_min_fuel=0.0, duration=0, valid=True)

        n_stations = sum(1 for s in segments if s.type == 'station')
        gap_segs   = [s for s in segments if s.type == 'caravan']
        n_gaps     = sum(s.gap_len for s in gap_segs)
        fuel_needed = sum(s.fuel_needed for s in gap_segs)
        duration = int(b_cap / max(fuel_needed, 0.1)) if fuel_needed > 0 else 0

        return dict(n_stations=n_stations, n_gaps=n_gaps,
                    fuel_needed=fuel_needed, bishop_min_fuel=b_cap,
                    duration=duration, valid=True)

    # ── Corridor construction ────────────────────────────────────

    def build_corridor(self, start: int, end: int,
                       kind: CorridorType = CorridorType.TEMPORARY,
                       n_pawns: int = 4, n_bishops: int = 1,
                       target: Target = None,
                       style: StyleCSS = None,
                       glyph: DiagramGlyphs = None) -> Corridor | None:
        existing = self.find_corridor(start, end)
        if existing:
            print(f"build_corridor: reusing ⬡{start}→⬡{end}")
            return existing

        p = self.profile
        pcr = pawn_covered_route(
            start, end, grid=self.grid, elevations=self.elevations,
            food_tiers=self.food_tiers,
            pawn_harvest=p.get('pawn_harvest', 1.3),
            pawn_diet=p.get('pawn_diet', 0.7),
            queen_diet=p.get('queen_diet', 7.0),
            cone_range=p.get('pawn_sight', 4),
            n_bishops=1,
            bishop_capacity=p.get('bishop_capacity', 140.0),
            bishop_diet=p.get('bishop_diet', 2.5),
            countries=self._countries())
        if pcr is None:
            print(f"build_corridor: no route ⬡{start}→⬡{end}")
            return None

        segments = find_stations_v3(pcr, n_pawns=n_pawns, n_bishops=1, profile=p)
        metrics = self._metrics_from_segments(segments)

        if n_bishops > 1 and metrics['fuel_needed'] > 0:
            metrics['bishop_min_fuel'] = p.get('bishop_capacity', 140.0) * n_bishops
            metrics['duration'] = int(metrics['bishop_min_fuel'] /
                                      max(metrics['fuel_needed'], 0.1))

        corridor = Corridor(hexes=pcr['route'], kind=kind,
                            segments=segments, target=target,
                            style=style, glyph=glyph, **metrics)
        self.corridors.append(corridor)

        tag = '✅' if corridor.feasible else '❌'
        print(f"Corridor ⬡{start}→⬡{end} ({kind.value}): "
              f"{corridor.length}hex, {corridor.n_stations} stations, "
              f"{corridor.n_gaps} gaps, fuel={corridor.fuel_needed:.1f} {tag}")
        return corridor

    # ── Bishop / congestion helpers ──────────────────────────────

    def assign_bishop(self, corridor: Corridor, bishop: Piece):
        fuel = getattr(bishop, 'food', self.profile.get('bishop_capacity', 140.0))
        corridor.bishop_min_fuel = min(corridor.bishop_min_fuel, fuel)

    def update_congestion(self):
        for c in self.corridors:
            rs = set(c.hexes)
            c.congestion = sum(
                1 for kingdom in self.board.kingdoms
                for settlement in kingdom.settlements
                for piece in settlement.citizens
                if piece.location in rs)

    # ── Crossing proposal ────────────────────────────────────────

    def propose_crossing(self, corridor: Corridor, queen: Piece,
                         available_pawns: int = 5,
                         available_bishops: int = 2,
                         max_retries: int = 3,
                         n_turns: int = 60) -> CrossingPlan:
        """Propose a validated crossing: assemble roster, simulate, retry."""
        camp = self._plan_camps(corridor, available_pawns)
        print(f"  Camp plan: {camp.summary()}")
        print(f"  Origin: {camp.origin}  Dest: {camp.dest}")

        best_plan = None
        route = corridor.hexes

        for attempt in range(max_retries + 1):
            extra_wait = attempt * 2

            # ── Assemble roster ──
            origin_r  = self._build_origin_shadows(camp, queen, route)
            advance_r = self._build_advance_shadows(camp, queen, route)
            station_r = self._build_station_shadows(camp, corridor, queen)
            bishop_r  = self._build_bishop_shadows(corridor, queen, available_bishops)

            queen_start = max(bishop_r.timing, advance_r.timing) + extra_wait
            qs = self._make_queen_shadow(queen, route, queen_start)

            shadows = ([qs] + origin_r.shadows + advance_r.shadows
                       + station_r.shadows + bishop_r.shadows)
            roster = ([dict(piece_type=PieceType.QUEEN, hex=route[0],
                            facing=qs.facing, role='queen')]
                      + origin_r.entries + advance_r.entries
                      + station_r.entries + bishop_r.entries)

            # ── Stamp ownership ──
            for s in shadows:
                s.owner_id = queen.owner_id
                s.flag = queen.flag

            # ── Simulate ──
            plan = CrossingPlan(
                corridor=corridor, roster=roster,
                queen_start_turn=queen_start)
            plan.simulate(
                shadows=shadows, queen_shadow=qs,
                grid=self.grid, elevations=self.elevations,
                food_tiers=self.food_tiers, profile=self.profile,
                countries=self._countries(), n_turns=n_turns)

            best_plan = plan

            if plan.feasible:
                roles = Counter(r['role'] for r in roster)
                print(f"✅ Crossing (attempt {attempt}): roles={dict(roles)}, "
                      f"start=t{queen_start}, arrive=t{plan.queen_arrival}, "
                      f"min_food={plan.food_margin:.1f}")
                return plan

            print(f"❌ Attempt {attempt}: arrive={'yes' if plan.queen_arrival < n_turns else 'no'}, "
                  f"min_food={plan.food_margin:.1f}, wait={queen_start}t")

        print(f"⚠️ All {max_retries + 1} attempts failed")
        return best_plan
```

```python
@patch
def simulate(self: CrossingPlan,
             shadows: list, queen_shadow,
             grid, elevations, food_tiers,
             profile: dict, countries=None,
             n_turns: int = 60) -> 'CrossingPlan':
    """Run FoodSimulator on shadows, populate result fields."""
    route = self.corridor.hexes

    shadow_squad = Squad.squads(1, queen_shadow.flag)[0]
    shadow_squad.pieces = list(shadows)

    initial = {s.id: dict(location=s.location, facing=s.facing,
                           food=s.food, health=s.health)
               for s in shadows}

    sim = FoodSimulator(
        grid=grid, elevations=elevations,
        food_tiers=food_tiers, pieces=shadows,
        countries=countries,
        squads=[shadow_squad],
        callbacks={shadow_squad.id: food_redistribute_adjacent})
    sim.run(num_turns=n_turns)

    # ── Evaluate queen trajectory ──
    q_snap = [s['piece_states'].get(queen_shadow.id) for s in sim.snapshots]
    alive   = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
    arrived = any(s and s['location'] == route[-1] for s in q_snap)
    min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                   default=0)

    arrival_turn = n_turns
    for i, s in enumerate(q_snap):
        if s and s['location'] == route[-1]:
            arrival_turn = i + 1
            break

    bishop_fuel = 0
    if sim.snapshots:
        for s in shadows:
            if s.piece_type == PieceType.BISHOP:
                ls = sim.snapshots[-1]['piece_states'].get(s.id)
                if ls:
                    bishop_fuel += profile['bishop_capacity'] - ls.get('food', 0)

    adapter = SimAdapter(sim, corridors=[self.corridor])
    self.schedule = adapter.extract(schedule_id=f'crossing',
                                     initial_states=initial)

    self.feasible = arrived and alive
    self.queen_arrival = arrival_turn
    self.food_margin = min_food
    self.bishop_fuel_used = bishop_fuel
    return self
```

Summary of changes:

| Before | After |
|---|---|
| 150-line monolith | ~30-line orchestrator + focused helpers |
| Dict-style `seg['type']` | Dataclass `seg.type` throughout |
| Inline PieceShadow construction ×5 | `_make_pawn_shadow` / `_make_queen_shadow` |
| Budget logic mixed with shadow creation | `CampPlan` + `_plan_camps` (testable alone) |
| Sim eval inside propose | `CrossingPlan.simulate()` owns its results |
| `RosterResult` groups (shadows, entries, timing) | Clean return from each builder |

In [ ]:
@patch
def simulate(self: CrossingPlan,
             shadows: list, queen_shadow,
             grid, elevations, food_tiers,
             profile: dict, countries=None,
             n_turns: int = 60) -> 'CrossingPlan':
    """Run FoodSimulator on shadows, populate result fields."""
    route = self.corridor.hexes

    shadow_squad = Squad.squads(1, queen_shadow.flag)[0]
    shadow_squad.pieces = list(shadows)

    initial = {s.id: dict(location=s.location, facing=s.facing,
                           food=s.food, health=s.health)
               for s in shadows}

    sim = FoodSimulator(
        grid=grid, elevations=elevations,
        food_tiers=food_tiers, pieces=shadows,
        countries=countries,
        squads=[shadow_squad],
        callbacks={shadow_squad.id: food_redistribute_adjacent})
    sim.run(num_turns=n_turns)

    # ── Evaluate queen trajectory ──
    q_snap = [s['piece_states'].get(queen_shadow.id) for s in sim.snapshots]
    alive   = q_snap[-1] and q_snap[-1]['health'] > 0 if q_snap else False
    arrived = any(s and s['location'] == route[-1] for s in q_snap)
    min_food = min((s['food'] for s in q_snap if s and s['health'] > 0),
                   default=0)

    arrival_turn = n_turns
    for i, s in enumerate(q_snap):
        if s and s['location'] == route[-1]:
            arrival_turn = i + 1
            break

    bishop_fuel = 0
    if sim.snapshots:
        for s in shadows:
            if s.piece_type == PieceType.BISHOP:
                ls = sim.snapshots[-1]['piece_states'].get(s.id)
                if ls:
                    bishop_fuel += profile['bishop_capacity'] - ls.get('food', 0)

    adapter = SimAdapter(sim, corridors=[self.corridor])
    self.schedule = adapter.extract(schedule_id=f'crossing',
                                     initial_states=initial)

    self.feasible = arrived and alive
    self.queen_arrival = arrival_turn
    self.food_margin = min_food
    self.bishop_fuel_used = bishop_fuel
    return self


In [ ]:
rebootWorld()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

c0, c1 = cities[0].location, cities[1].location
print(f"Cities: ⬡{c0} → ⬡{c1}\n")

# 2. Build coverage + corridor
qc = QueenCoverage(
    board=myStuff.board, grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers, profile=FOOD_PROFILE_V2,
    queens=[myStuff.queen],
)

corridor = qc.build_corridor(c0, c1, kind=CorridorType.PERMANENT,
                              n_pawns=5, n_bishops=2)
print(f"\nCorridor: {corridor.length} hexes, "
      f"{corridor.n_stations} stations, {corridor.n_gaps} gaps, "
      f"feasible={'✅' if corridor.feasible else '❌'}")

if not corridor.feasible:
    print("❌ Corridor not feasible — can't test crossing")
else:
    # 3. Teleport queen to corridor start, fill her up
    queen = myStuff.queen
    queen.location = corridor.hexes[0]
    apply_food_profile([queen])
    print(f"\nQueen {queen.name} at ⬡{queen.location}, "
          f"food={queen.food:.1f}, diet={queen.diet}")

    # 4. Propose crossing
    print(f"\n{'='*60}")
    print("Proposing crossing plan...")
    plan = qc.propose_crossing(
        corridor, queen,
        available_pawns=5,
        available_bishops=2,
        n_turns=60,
        max_retries=3,
    )

    # 5. Results
    print(f"\n{'='*60}")
    print(f"Feasible:     {plan.feasible}")
    print(f"Pieces:       {dict(plan.pieces_needed())}")
    print(f"Queen starts: turn {plan.queen_start_turn}")
    print(f"Arrival:      turn {plan.queen_arrival}")
    print(f"Food margin:  {plan.food_margin:.1f}")
    print(f"Bishop fuel:  {plan.bishop_fuel_used:.0f}")

    # 6. Show the plan card + timetable
    show(plan)

I am wondering about a different kind of schedule where we have the hex locations at the top time as the rows and then who is there as the fields. we would want a sensible ordering of the columns so that there is some sort of adjacency (though it will have to break since we are projecting 2d space onto 1d)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love this idea — it's essentially a **space-time diagram**. The corridor gives you a natural 1D backbone for column ordering:

1. **Route hexes** in corridor order (the spine)
2. **Off-route hexes** (camps, bishop parks) slotted next to their nearest route hex

For cell content: piece initials + action icon, colored by role.

```python
def _hex_sort_key(hex_idx, corridor):
    """Project a hex onto the corridor's 1D route order.
    
    On-route hexes get integer position.
    Off-route hexes get (nearest_route_pos ± 0.5) so they sit adjacent.
    """
    if corridor and hex_idx in corridor:
        return (corridor.index(hex_idx), 0)  # on-route, primary
    
    # Off-route: find nearest route hex
    if corridor:
        grid = None  # we'll use route index proximity
        best_w, best_d = 0, float('inf')
        for w, rh in enumerate(corridor.hexes):
            # Cheap proxy: just use route index distance isn't right...
            # We need grid distance, but we may not have it here.
            # Fallback: treat unknown hexes as pre-route or post-route
            if hex_idx == rh:
                return (w, 0)
        # Can't resolve — put at end
        return (len(corridor.hexes), hex_idx)
    
    return (0, hex_idx)


def space_time_timetable(df, corridor=None, grid=None,
                         title='Space-Time', max_rows=40):
    """Time×Space schedule: rows=turns, columns=hex locations, cells=occupants.
    
    Column ordering: corridor route hexes in sequence, off-route hexes
    slotted next to their nearest route neighbor.
    """
    if df.empty:
        return Card(P('Empty schedule'), header=CardTitle(title))

    turns = sorted(df.turn.unique())
    if len(turns) > max_rows:
        step = max(1, len(turns) // max_rows)
        turns = turns[::step]

    all_hexes = sorted(df.hex.unique())

    # ── Column ordering ──
    if corridor and grid:
        route_set = set(corridor.hexes)

        def _sort_key(h):
            if h in route_set:
                return (corridor.index(h), 0)
            # Off-route: find nearest route hex by grid distance
            hp_h = grid.index_to_hexposition(h)
            best_w, best_d = 0, 999
            for w, rh in enumerate(corridor.hexes):
                hp_r = grid.index_to_hexposition(rh)
                d = hp_h.distance(hp_r)
                if d < best_d:
                    best_d = d
                    best_w = w
            return (best_w, 1)  # 1 = off-route sorts after on-route at same position

        all_hexes = sorted(all_hexes, key=_sort_key)
    elif corridor:
        # No grid — just use membership + index
        route_set = set(corridor.hexes)
        on_route = [h for h in corridor.hexes if h in set(all_hexes)]
        off_route = sorted(h for h in all_hexes if h not in route_set)
        all_hexes = on_route + off_route

    # ── Role → abbreviation + color ──
    _ROLE_ABBR = {
        'queen': ('Q', '#8e44ad'),
        'origin_camp': ('O', '#27ae60'),
        'advance_party': ('A', '#2980b9'),
        'station': ('S', '#f39c12'),
        'lighthouse': ('B', '#c0392b'),
        'escort': ('B', '#e74c3c'),
    }

    # Build piece_id → (abbr, color) lookup from roster info in schedule
    piece_meta = {}
    for _, row in df.drop_duplicates('piece_id').iterrows():
        name = row.get('piece_name', '?')
        # Try to infer role from piece name patterns
        for role_key, (abbr, color) in _ROLE_ABBR.items():
            if role_key in name.lower() or (role_key == 'queen' and 'queen' in name.lower()):
                piece_meta[row.piece_id] = (name[:3], color)
                break
        else:
            piece_meta[row.piece_id] = (name[:3], '#555')

    # ── Header: hex labels ──
    route_set = set(corridor.hexes) if corridor else set()
    hex_headers = []
    for h in all_hexes:
        on = h in route_set
        style = 'font-weight:bold' if on else 'opacity:0.6;font-style:italic'
        hex_headers.append(Th(f'⬡{h}', style=f'{style};font-size:0.75em;writing-mode:vertical-lr'))

    header = Tr(Th('t', style='width:2em'), *hex_headers)

    # ── Rows: one per turn ──
    hex_to_col = {h: i for i, h in enumerate(all_hexes)}
    rows = []

    for t in turns:
        t_data = df[df.turn == t]
        cells = [Td('', style='background:#fafafa') for _ in all_hexes]

        for _, row in t_data.iterrows():
            h = row.hex
            if h not in hex_to_col: continue
            ci = hex_to_col[h]

            pid = row.piece_id
            abbr, color = piece_meta.get(pid, ('?', '#555'))
            food = row.get('food', 0)
            action = row.get('action', '—')
            tip = f"{row.get('piece_name', '?')} f={food:.0f} {action}"

            # Stack multiple occupants
            existing = cells[ci]
            if hasattr(existing, 'children') and existing.children:
                # Append to existing cell
                prev = existing.children[0] if existing.children else ''
                cells[ci] = Td(
                    NotStr(f'{prev}<br>'),
                    Span(abbr, style=f'color:{color};font-weight:bold;font-size:0.8em',
                         title=tip),
                    style='background:#fff8e1' if len(str(prev)) > 2 else '')
            else:
                cells[ci] = Td(
                    Span(abbr, style=f'color:{color};font-weight:bold;font-size:0.8em',
                         title=tip))

        rows.append(Tr(
            Td(Strong(f'{int(t)}'), style='font-size:0.8em'),
            *cells))

    return Card(
        Table(Thead(header), Tbody(*rows),
              cls=TableT.sm,
              style='border-collapse:collapse'),
        header=CardTitle(title),
        style='overflow-x:auto',
    )
```

Key design decisions:

- **Column ordering**: on-route hexes in corridor sequence, off-route hexes slotted by nearest route neighbor (with grid distance if available, index proximity otherwise)
- **On-route columns are bold**, off-route are italic/dimmed — so you can instantly see the corridor spine
- **Cells**: role-colored abbreviation (Q=queen, O=origin, A=advance, S=station, B=bishop), with hover tooltip showing full name + food + action
- **Congestion**: multiple occupants stack vertically in the same cell, cell gets highlighted background
- **Vertical hex labels**: saves horizontal space since there could be 20+ columns

You'd call it like:

```python
space_time_timetable(plan.schedule, corridor=corridor, grid=myStuff.grid)
```

In [ ]:
#| export
# ── Role config: emoji + color ──
_ROLE_CFG = {
    'queen':         ('👑', '#8e44ad', 'Queen'),
    'origin_camp':   ('🌾', '#27ae60', 'Origin Camp'),
    'advance_party': ('🏃', '#2980b9', 'Advance Party'),
    'station':       ('⛺', '#f39c12', 'Station'),
    'lighthouse':    ('🔦', '#c0392b', 'Lighthouse Bishop'),
    'escort':        ('🐫', '#e74c3c', 'Escort Bishop'),
}

def _role_legend():
    """Horizontal legend strip for space-time timetable."""
    items = []
    for role, (emoji, color, label) in _ROLE_CFG.items():
        items.append(Span(
            Span(emoji, style='font-size:1.1em'),
            Span(f' {label}', style=f'color:{color};font-weight:bold;font-size:0.8em'),
            style='margin-right:1em;white-space:nowrap',
        ))
    return Div(*items, style='display:flex;flex-wrap:wrap;gap:0.25em 0;padding:0.5em 0',
               cls='uk-text-muted')

def _hex_sort_key(hex_idx, corridor):
    """Project a hex onto the corridor's 1D route order.
    
    On-route hexes get integer position.
    Off-route hexes get (nearest_route_pos ± 0.5) so they sit adjacent.
    """
    if corridor and hex_idx in corridor:
        return (corridor.index(hex_idx), 0)  # on-route, primary
    
    # Off-route: find nearest route hex
    if corridor:
        grid = None  # we'll use route index proximity
        best_w, best_d = 0, float('inf')
        for w, rh in enumerate(corridor.hexes):
            # Cheap proxy: just use route index distance isn't right...
            # We need grid distance, but we may not have it here.
            # Fallback: treat unknown hexes as pre-route or post-route
            if hex_idx == rh:
                return (w, 0)
        # Can't resolve — put at end
        return (len(corridor.hexes), hex_idx)
    
    return (0, hex_idx)


def space_time_timetable(df, corridor=None, grid=None,
                         roster=None, title='Space-Time', max_rows=40):
    """Time×Space schedule: rows=turns, columns=hex locations, cells=occupants.
    
    Args:
        roster: list of dicts from CrossingPlan.roster — used for role→emoji mapping.
    """
    if df.empty:
        return Card(P('Empty schedule'), header=CardTitle(title))

    turns = sorted(df.turn.unique())
    if len(turns) > max_rows:
        step = max(1, len(turns) // max_rows)
        turns = turns[::step]

    all_hexes = sorted(df.hex.unique())

    # ── Column ordering ──
    if corridor and grid:
        route_set = set(corridor.hexes)
        def _sort_key(h):
            if h in route_set:
                return (corridor.index(h), 0)
            hp_h = grid.index_to_hexposition(h)
            best_w, best_d = 0, 999
            for w, rh in enumerate(corridor.hexes):
                hp_r = grid.index_to_hexposition(rh)
                d = hp_h.distance(hp_r)
                if d < best_d:
                    best_d = d
                    best_w = w
            return (best_w, 1)
        all_hexes = sorted(all_hexes, key=_sort_key)
    elif corridor:
        route_set = set(corridor.hexes)
        on_route = [h for h in corridor.hexes if h in set(all_hexes)]
        off_route = sorted(h for h in all_hexes if h not in route_set)
        all_hexes = on_route + off_route

    # ── Build piece_id → (emoji, color) from roster ──
    piece_meta = {}
    if roster:
        # Roster entries have piece_type + role; match to schedule by position
        # Build name→role from roster naming convention
        role_names = {}
        queen_name = None
        for r in roster:
            role = r.get('role', '')
            if role == 'queen':
                queen_name = True  # first piece in schedule is queen
            # We'll match by piece_name patterns below
            
    # Fallback: infer from piece_name in schedule
    for _, row in df.drop_duplicates('piece_id').iterrows():
        name = str(row.get('piece_name', ''))
        pid = row.piece_id
        matched = False
        
        # Match patterns to roles
        patterns = [
            ('camp_o', 'origin_camp'),
            ('camp_d', 'advance_party'),
            ('origin', 'origin_camp'),
            ('advance', 'advance_party'),
            ('station', 'station'),
            ('blh_', 'lighthouse'),
            ('besc_', 'escort'),
            ('bishop', 'lighthouse'),
        ]
        for pat, role in patterns:
            if pat in name.lower():
                emoji, color, _ = _ROLE_CFG[role]
                piece_meta[pid] = (emoji, color)
                matched = True
                break
        
        if not matched:
            # Check if it's the queen (first piece, or name matches)
            if any(t in name.lower() for t in ('queen',)) or row.get('piece_name', '') not in [
                r.get('piece_name') for r in (roster or []) if r.get('role') != 'queen']:
                # Default: check if piece_type info available
                emoji, color, _ = _ROLE_CFG.get('queen', ('❓', '#555', '?'))
                piece_meta[pid] = (emoji, color)
    
    # Simpler: if roster provided, use index alignment
    if roster:
        piece_ids_ordered = list(df.drop_duplicates('piece_id').piece_id)
        # Roster order matches shadow order: queen first, then camps, etc.
        for i, r in enumerate(roster):
            if i < len(piece_ids_ordered):
                role = r.get('role', '')
                if role in _ROLE_CFG:
                    emoji, color, _ = _ROLE_CFG[role]
                    piece_meta[piece_ids_ordered[i]] = (emoji, color)

    # ── Header: hex labels ──
    route_set = set(corridor.hexes) if corridor else set()
    hex_headers = []
    for h in all_hexes:
        on = h in route_set
        style = 'font-weight:bold' if on else 'opacity:0.6;font-style:italic'
        hex_headers.append(Th(f'⬡{h}', style=f'{style};font-size:0.75em;writing-mode:vertical-lr'))

    header = Tr(Th('t', style='width:2em'), *hex_headers)

    # ── Rows: one per turn ──
    hex_to_col = {h: i for i, h in enumerate(all_hexes)}
    rows = []

    for t in turns:
        t_data = df[df.turn == t]
        cells = [''] * len(all_hexes)  # accumulate strings per cell

        for _, row in t_data.iterrows():
            h = row.hex
            if h not in hex_to_col: continue
            ci = hex_to_col[h]

            pid = row.piece_id
            emoji, color = piece_meta.get(pid, ('❓', '#555'))
            food = row.get('food', 0)
            action = row.get('action', '—')
            name = row.get('piece_name', '?')
            tip = f"{name} f={food:.0f} {action}"

            cells[ci] += f'<span title="{tip}" style="cursor:help">{emoji}</span>'

        row_cells = []
        for content in cells:
            if content:
                bg = '#fff8e1' if content.count('<span') > 1 else ''
                style = f'background:{bg};text-align:center' if bg else 'text-align:center'
                row_cells.append(Td(NotStr(content), style=style))
            else:
                row_cells.append(Td('', style='background:#fafafa'))

        rows.append(Tr(
            Td(Strong(f'{int(t)}'), style='font-size:0.8em'),
            *row_cells))

    return Div(
        Card(
            Table(Thead(header), Tbody(*rows),
                  cls=TableT.sm,
                  style='border-collapse:collapse'),
            header=CardTitle(title),
            style='overflow-x:auto',
        ),
        _role_legend(),
    )

In [ ]:
space_time_timetable(plan.schedule, corridor=corridor, grid=myStuff.grid)

How did we have so many pieces at the same hex at the start

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's by design in the simulation — **all shadows spawn at `route[0]`** (⬡158) because they represent the queen's army departing from her position:

- **Queen** → `qs.location = route[0]`
- **Advance party** → `self._make_pawn_shadow(route[0], 0, ...)` — they sprint *from* start to dest camps
- **Bishops** → `_make_bishop_shadow` starts at `route[0]` and sprints to their lighthouse/escort positions

Only the **origin camp pawns** and **station pawns** start at their actual hex (⬡179, ⬡159, ⬡248) because they're pre-positioned.

That's why turn 0 shows 👑🏃🏃🔦🔦 all stacked at ⬡158 — they haven't fanned out yet. By turn 1-2 you can see the 🏃 and 🔦 spreading to their destinations.

The stacking also causes the `BUMPED`/`BLOCKED` actions in early turns — pieces are colliding as they try to leave the same hex simultaneously. That's a known artifact of the shadow simulation (real deployment would stagger spawning).

If you wanted to clean up the visual, you could either:
- **Stagger spawn turns** — bishops depart turn 0, advance party turn 1, queen later
- **Pre-place at destinations** — skip the sprint simulation for pieces whose travel is "assumed" (but then you lose the fuel accounting)